# 03 – Feature Engineering

## Project: Netflix Customer Intelligence & AI Analytics Copilot

### Objective
Transform the cleaned Netflix datasets into meaningful customer-level features for Machine Learning.

### Key Activities
- Customer feature creation
- Viewing behaviour analysis
- Subscription feature creation
- Payment behaviour analysis
- Customer support and feedback analysis
- Customer-level feature aggregation
- Feature preparation for Machine Learning

### Data Source
`data/processed/final/`

### Tools
Python, Pandas, NumPy, Scikit-learn

In [2]:
# Find if foldeer exists or not
import pandas as pd
import numpy as np
from pathlib import Path

project_root = Path.cwd().parent

final_data_path = project_root / "data" / "processed" / "final"

print("Project root:", project_root)
print("Final data folder:", final_data_path)
print("Final data folder exists:", final_data_path.exists())

Project root: c:\Users\USER\Documents\Internmo Project1\fae5dd037f3e68b4f411b2243b81c23d
Final data folder: c:\Users\USER\Documents\Internmo Project1\fae5dd037f3e68b4f411b2243b81c23d\data\processed\final
Final data folder exists: True


In [3]:
# Load all nine datasets
cleaned_datasets = {
    "customers": pd.read_csv(final_data_path / "customers_final.csv"),
    "subscription_plans": pd.read_csv(final_data_path / "subscription_plans_final.csv"),
    "content": pd.read_csv(final_data_path / "content_final.csv"),
    "subscriptions": pd.read_csv(final_data_path / "subscriptions_final1.csv"),
    "viewing_activity": pd.read_csv(final_data_path / "viewing_activity_final.csv"),
    "payments": pd.read_csv(final_data_path / "payments_final1.csv"),
    "support_tickets": pd.read_csv(final_data_path / "support_tickets_final1.csv"),
    "customer_feedback": pd.read_csv(final_data_path / "customer_feedback_final1.csv"),
    "churn_labels": pd.read_csv(final_data_path / "churn_labels_final.csv")
}

print("All datasets loaded successfully!")
print("Total datasets:", len(cleaned_datasets))

All datasets loaded successfully!
Total datasets: 9


In [4]:
# step 4
dataset_shapes = []

for name, df in cleaned_datasets.items():
    dataset_shapes.append({
        "Dataset": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1]
    })

dataset_summary = pd.DataFrame(dataset_shapes)

display(dataset_summary)

,Dataset,Rows,Columns
0,customers,8000,12
1,subscription_plans,3,6
2,content,500,9
3,subscriptions,8875,10
4,viewing_activity,66422,9
5,payments,110735,8
6,support_tickets,6120,11
7,customer_feedback,5046,6
8,churn_labels,8000,4


In [5]:
# Step 5
for name, df in cleaned_datasets.items():
    print(f"\n{'=' * 50}")
    print(f"Dataset: {name}")
    print(f"{'=' * 50}")

    print(df.dtypes)


Dataset: customers
customer_id                str
first_name                 str
last_name                  str
age                      int64
gender                     str
country                    str
state_or_region        float64
city                       str
registration_date          str
acquisition_channel        str
customer_segment           str
preferred_language         str
dtype: object

Dataset: subscription_plans
plan_id               str
plan_name             str
monthly_price     float64
video_quality         str
max_devices         int64
advertisements        str
dtype: object

Dataset: content
content_id              str
title                   str
content_type            str
genre                   str
release_year          int64
duration_minutes      int64
content_language        str
production_country      str
maturity_rating         str
dtype: object

Dataset: subscriptions
subscription_id                str
customer_id                    str
plan_id          

In [6]:
# Step 6: Identify date and time columns
date_keywords = [
    "date", "time", "created", "updated",
    "closed", "renewal", "start", "end"
]

date_columns = []

for name, df in cleaned_datasets.items():
    for column in df.columns:
        if any(keyword in column.lower() for keyword in date_keywords):
            date_columns.append({
                "Dataset": name,
                "Column": column,
                "Data Type": str(df[column].dtype)
            })

date_columns_summary = pd.DataFrame(date_columns)

display(date_columns_summary)

,Dataset,Column,Data Type
0,customers,gender,str
1,customers,registration_date,str
2,subscriptions,subscription_start_date,str
3,subscriptions,subscription_end_date,str
4,subscriptions,cancellation_date,str
5,viewing_activity,viewing_date,str
6,payments,payment_date,str
7,support_tickets,ticket_date,str
8,support_tickets,resolution_time_hours,float64
9,customer_feedback,feedback_date,str


In [7]:
# Step 6.1 – Filter only actual date columns
date_columns = []

for name, df in cleaned_datasets.items():
    for column in df.columns:
        if column.lower().endswith("_date"):
            date_columns.append({
                "Dataset": name,
                "Column": column,
                "Data Type": str(df[column].dtype)
            })

date_columns_summary = pd.DataFrame(date_columns)

display(date_columns_summary)

,Dataset,Column,Data Type
0,customers,registration_date,str
1,subscriptions,subscription_start_date,str
2,subscriptions,subscription_end_date,str
3,subscriptions,cancellation_date,str
4,viewing_activity,viewing_date,str
5,payments,payment_date,str
6,support_tickets,ticket_date,str
7,customer_feedback,feedback_date,str
8,churn_labels,churn_date,str


In [8]:
# Step 7 – Convert date columns into datetime format
date_column_mapping = {
    "customers": ["registration_date"],
    "subscriptions": [
        "subscription_start_date",
        "subscription_end_date",
        "cancellation_date"
    ],
    "viewing_activity": ["viewing_date"],
    "payments": ["payment_date"],
    "support_tickets": ["ticket_date"],
    "customer_feedback": ["feedback_date"],
    "churn_labels": ["churn_date"]
}

for dataset_name, columns in date_column_mapping.items():
    for column in columns:
        cleaned_datasets[dataset_name][column] = pd.to_datetime(
            cleaned_datasets[dataset_name][column],
            errors="coerce"
        )

print("Date columns converted successfully!")

Date columns converted successfully!


In [9]:
for dataset_name, columns in date_column_mapping.items():
    print(f"\nDataset: {dataset_name}")

    for column in columns:
        print(
            column,
            ":", cleaned_datasets[dataset_name][column].dtype
        )


Dataset: customers
registration_date : datetime64[us]

Dataset: subscriptions
subscription_start_date : datetime64[us]
subscription_end_date : datetime64[us]
cancellation_date : datetime64[us]

Dataset: viewing_activity
viewing_date : datetime64[us]

Dataset: payments
payment_date : datetime64[us]

Dataset: support_tickets
ticket_date : datetime64[us]

Dataset: customer_feedback
feedback_date : datetime64[us]

Dataset: churn_labels
churn_date : datetime64[us]


In [10]:
# Step 8 – Validate converted dates
date_validation = []

for dataset_name, columns in date_column_mapping.items():
    for column in columns:
        date_series = cleaned_datasets[dataset_name][column]

        date_validation.append({
            "Dataset": dataset_name,
            "Column": column,
            "Missing Dates": date_series.isna().sum(),
            "Earliest Date": date_series.min(),
            "Latest Date": date_series.max()
        })

date_validation_df = pd.DataFrame(date_validation)

display(date_validation_df)

,Dataset,Column,Missing Dates,Earliest Date,Latest Date
0,customers,registration_date,0,2019-01-01,2026-05-31
1,subscriptions,subscription_start_date,0,2019-01-01,2026-06-29
2,subscriptions,subscription_end_date,6051,2019-04-11,2026-06-30
3,subscriptions,cancellation_date,6884,2025-10-03,2026-06-30
4,viewing_activity,viewing_date,0,2025-07-01,2026-06-30
5,payments,payment_date,0,2019-01-01,2026-05-31
6,support_tickets,ticket_date,61,2019-03-03,2026-06-30
7,customer_feedback,feedback_date,0,2019-03-06,2026-06-30
8,churn_labels,churn_date,6009,2025-10-03,2026-06-30


In [11]:
# Investigate the 61 missing support ticket dates
missing_ticket_dates = cleaned_datasets["support_tickets"][
    cleaned_datasets["support_tickets"]["ticket_date"].isna()
]

print("Records with missing ticket dates:", len(missing_ticket_dates))

display(missing_ticket_dates)

Records with missing ticket dates: 61


,ticket_id,customer_id,ticket_date,issue_category,issue_subcategory,priority,ticket_status,resolution_time_hours,customer_satisfaction_score,ticket_description,support_agent_id
167,TCK0000168,CUST000196,NaT,Content,Content unavailable in region,Medium,Escalated,NaN,NaN,Several titles I could watch last month are no...,AGT0015
233,TCK0000234,CUST000293,NaT,Device & App,App not loading,Low,Resolved,21.9,4.0,The app does not load past the splash screen o...,AGT0044
296,TCK0000297,CUST000372,NaT,Content,Missing subtitles,High,Resolved,4.6,3.0,Subtitles are missing for several titles in my...,AGT0058
319,TCK0000320,CUST000397,NaT,Streaming Quality,Audio sync issue,Medium,Resolved,14.7,1.0,The audio is out of sync with the video on alm...,AGT0044
346,TCK0000347,CUST000428,NaT,Billing & Payments,Price dispute,Critical,Pending,NaN,NaN,I did not receive any communication about the ...,AGT0105
...,...,...,...,...,...,...,...,...,...,...,...
5153,TCK0005154,CUST006716,NaT,Account Access,Account locked,Low,Resolved,48.7,2.0,My account got locked after multiple login att...,AGT0089
5164,TCK0005165,CUST006737,NaT,Device & App,Device not supported,Low,Resolved,0.5,3.0,My new smart TV model shows the app as unsuppo...,AGT0075
5339,TCK0005340,CUST006968,NaT,Subscription,Auto-renew complaint,High,Pending,NaN,NaN,I was auto renewed without any reminder notifi...,AGT0013
5446,TCK0005447,CUST007133,NaT,Subscription,Plan upgrade request,Low,Resolved,30.8,5.0,I want to upgrade to the premium plan for bett...,AGT0023


In [12]:
# Step 9 – Establish the dataset reference date
snapshot_date = pd.Timestamp("2026-06-30")

print("Dataset snapshot date:", snapshot_date)

Dataset snapshot date: 2026-06-30 00:00:00


In [13]:
# Step 10 – Customer Features
customer_features = cleaned_datasets["customers"].copy()

customer_features["age_group"] = pd.cut(
    customer_features["age"],
    bins=[0, 17, 24, 34, 44, 54, np.inf],
    labels=["Under 18", "18-24", "25-34", "35-44", "45-54", "55+"]
)

customer_features["customer_tenure_days"] = (
    snapshot_date - customer_features["registration_date"]
).dt.days

display(
    customer_features[
        ["customer_id", "age", "age_group", "registration_date", "customer_tenure_days"]
    ].head(10)
)

,customer_id,age,age_group,registration_date,customer_tenure_days
0,CUST000744,21,18-24,2019-06-03,2584
1,CUST001502,31,25-34,2026-01-13,168
2,CUST005383,40,35-44,2023-06-17,1109
3,CUST004708,27,25-34,2019-03-09,2670
4,CUST005877,36,35-44,2025-12-03,209
5,CUST003762,29,25-34,2025-10-12,261
6,CUST007892,31,25-34,2025-08-31,303
7,CUST002317,38,35-44,2020-04-21,2261
8,CUST005609,38,35-44,2023-01-03,1274
9,CUST000519,29,25-34,2024-02-07,874


In [14]:
# Step 11 – Validate Customer Features
age_group_distribution = (
    customer_features["age_group"]
    .value_counts(dropna=False)
    .rename_axis("Age Group")
    .reset_index(name="Customer Count")
)

display(age_group_distribution)

print("Missing age groups:", customer_features["age_group"].isna().sum())
print("Missing tenure values:", customer_features["customer_tenure_days"].isna().sum())
print("Negative tenure values:", (customer_features["customer_tenure_days"] < 0).sum())

display(customer_features["customer_tenure_days"].describe())

,Age Group,Customer Count
0,25-34,2927
1,35-44,2191
2,18-24,1136
3,45-54,878
4,Under 18,684
5,55+,184


Missing age groups: 0
Missing tenure values: 0
Negative tenure values: 0


count    8000.000000
mean     1375.986250
std       776.164124
min        30.000000
25%       706.750000
50%      1376.000000
75%      2040.000000
max      2737.000000
Name: customer_tenure_days, dtype: float64

In [15]:
# Step 12 – Subscription Features
display(cleaned_datasets["subscriptions"].head())

print("Columns:")
print(cleaned_datasets["subscriptions"].columns.tolist())

,subscription_id,customer_id,plan_id,subscription_start_date,subscription_end_date,subscription_status,auto_renew,cancellation_date,cancellation_reason,monthly_price
0,SUB0000001,CUST000001,PLAN03,2022-04-30,NaT,Active,True,NaT,NaN,22.99
1,SUB0000002,CUST000002,PLAN02,2025-12-24,NaT,Active,True,NaT,NaN,15.49
2,SUB0000003,CUST000003,PLAN03,2019-12-15,2026-04-20,Cancelled,True,2026-04-20,Too expensive,22.99
3,SUB0000004,CUST000004,PLAN01,2019-01-28,NaT,Active,False,NaT,NaN,6.99
4,SUB0000005,CUST000005,PLAN01,2024-07-24,NaT,Active,True,NaT,NaN,6.99


Columns:
['subscription_id', 'customer_id', 'plan_id', 'subscription_start_date', 'subscription_end_date', 'subscription_status', 'auto_renew', 'cancellation_date', 'cancellation_reason', 'monthly_price']


In [16]:
# Step 13 – Inspect Subscription Values
subscriptions = cleaned_datasets["subscriptions"]

print("Subscription Status:")
display(subscriptions["subscription_status"].value_counts(dropna=False))

print("Auto-Renewal Distribution:")
display(subscriptions["auto_renew"].value_counts(dropna=False))

print("Cancellation Reasons:")
display(subscriptions["cancellation_reason"].value_counts(dropna=False))

print("Missing Values:")
display(subscriptions.isna().sum())

Subscription Status:


subscription_status
Active          6009
Cancelled       1991
Plan Changed     875
Name: count, dtype: int64

Auto-Renewal Distribution:


auto_renew
True     7227
False    1648
Name: count, dtype: int64

Cancellation Reasons:


cancellation_reason
NaN                                 6884
Too expensive                        365
Found alternative platform           284
Not enough content                   217
No longer needed                     212
Technical issues                     182
Household budget cuts                145
Content quality decline              139
Payment issues                       139
Poor customer service experience     127
Moving / relocation                   98
Password sharing crackdown            83
Name: count, dtype: int64

Missing Values:


subscription_id               0
customer_id                   0
plan_id                       0
subscription_start_date       0
subscription_end_date      6051
subscription_status           0
auto_renew                    0
cancellation_date          6884
cancellation_reason        6884
monthly_price                 0
dtype: int64

In [17]:
print("Subscription Status:")
display(subscriptions["subscription_status"].value_counts(dropna=False))

print("Auto-Renewal Distribution:")
display(subscriptions["auto_renew"].value_counts(dropna=False))

print("Remaining Missing Values:")
display(subscriptions[["cancellation_reason", "monthly_price"]].isna().sum())

Subscription Status:


subscription_status
Active          6009
Cancelled       1991
Plan Changed     875
Name: count, dtype: int64

Auto-Renewal Distribution:


auto_renew
True     7227
False    1648
Name: count, dtype: int64

Remaining Missing Values:


cancellation_reason    6884
monthly_price             0
dtype: int64

In [18]:
print("Subscription Status:")
display(subscriptions["subscription_status"].value_counts())

print("Auto-Renewal Distribution:")
display(subscriptions["auto_renew"].value_counts())

Subscription Status:


subscription_status
Active          6009
Cancelled       1991
Plan Changed     875
Name: count, dtype: int64

Auto-Renewal Distribution:


auto_renew
True     7227
False    1648
Name: count, dtype: int64

In [19]:
# Step 14 – Validate Subscription Dates by Status
subscription_date_validation = subscriptions.groupby("subscription_status").agg(
    Total_Subscriptions=("subscription_id", "count"),
    Available_End_Dates=("subscription_end_date", "count"),
    Missing_End_Dates=("subscription_end_date", lambda x: x.isna().sum()),
    Available_Cancellation_Dates=("cancellation_date", "count"),
    Missing_Cancellation_Dates=("cancellation_date", lambda x: x.isna().sum())
)

display(subscription_date_validation)

,Total_Subscriptions,Available_End_Dates,Missing_End_Dates,Available_Cancellation_Dates,Missing_Cancellation_Dates
subscription_status,,,,,
Active,6009,0,6009,0,6009
Cancelled,1991,1949,42,1991,0
Plan Changed,875,875,0,0,875


In [20]:
# Step 15 – Calculate Subscription Duration
subscription_features = subscriptions.copy()

subscription_features["effective_end_date"] = (
    subscription_features["subscription_end_date"]
    .fillna(subscription_features["cancellation_date"])
    .fillna(snapshot_date)
)

subscription_features["subscription_duration_days"] = (
    subscription_features["effective_end_date"]
    - subscription_features["subscription_start_date"]
).dt.days

display(
    subscription_features[
        [
            "customer_id",
            "subscription_status",
            "subscription_start_date",
            "subscription_end_date",
            "cancellation_date",
            "effective_end_date",
            "subscription_duration_days"
        ]
    ].head(10)
)

,customer_id,subscription_status,subscription_start_date,subscription_end_date,cancellation_date,effective_end_date,subscription_duration_days
0,CUST000001,Active,2022-04-30,NaT,NaT,2026-06-30,1522
1,CUST000002,Active,2025-12-24,NaT,NaT,2026-06-30,188
2,CUST000003,Cancelled,2019-12-15,2026-04-20,2026-04-20,2026-04-20,2318
3,CUST000004,Active,2019-01-28,NaT,NaT,2026-06-30,2710
4,CUST000005,Active,2024-07-24,NaT,NaT,2026-06-30,706
5,CUST000006,Active,2025-07-19,NaT,NaT,2026-06-30,346
6,CUST000007,Cancelled,2025-10-30,2026-05-05,2026-05-05,2026-05-05,187
7,CUST000008,Active,2023-08-21,NaT,NaT,2026-06-30,1044
8,CUST000009,Active,2019-11-18,NaT,NaT,2026-06-30,2416
9,CUST000010,Active,2021-12-03,NaT,NaT,2026-06-30,1670


In [21]:
# Step 16 – Validate Subscription Duration
print("Missing Duration Values:",subscription_features["subscription_duration_days"].isna().sum())

print("Negative Duration Values:",(subscription_features["subscription_duration_days"] < 0).sum())

duration_summary = subscription_features.groupby("subscription_status")["subscription_duration_days"].agg(["count", "mean", "median", "min", "max"])

display(duration_summary)

Missing Duration Values: 0
Negative Duration Values: 42


,count,mean,median,min,max
subscription_status,,,,,
Active,6009,1329.270261,1330.0,1,2737
Cancelled,1991,1072.083375,966.0,-188,2713
Plan Changed,875,716.484571,554.0,62,2653


In [22]:
negative_duration_records = subscription_features[
    subscription_features["subscription_duration_days"] < 0
]

display(
    negative_duration_records[
        [
            "subscription_id",
            "customer_id",
            "subscription_status",
            "subscription_start_date",
            "subscription_end_date",
            "cancellation_date",
            "effective_end_date",
            "subscription_duration_days"
        ]
    ]
)

print("Total negative duration records:",
      len(negative_duration_records))

,subscription_id,customer_id,subscription_status,subscription_start_date,subscription_end_date,cancellation_date,effective_end_date,subscription_duration_days
125,SUB0000126,CUST000113,Cancelled,2026-04-18,NaT,2025-12-09,2025-12-09,-130
157,SUB0000158,CUST000143,Cancelled,2026-03-02,NaT,2025-12-27,2025-12-27,-65
301,SUB0000302,CUST000268,Cancelled,2026-03-02,NaT,2026-01-29,2026-01-29,-32
331,SUB0000332,CUST000296,Cancelled,2026-01-26,NaT,2025-12-07,2025-12-07,-50
539,SUB0000540,CUST000491,Cancelled,2026-06-28,NaT,2025-12-22,2025-12-22,-188
735,SUB0000736,CUST000670,Cancelled,2026-01-03,NaT,2025-12-02,2025-12-02,-32
940,SUB0000941,CUST000859,Cancelled,2026-01-26,NaT,2025-10-08,2025-10-08,-110
1024,SUB0001025,CUST000935,Cancelled,2026-02-24,NaT,2025-11-23,2025-11-23,-93
1282,SUB0001283,CUST001173,Cancelled,2026-04-09,NaT,2025-11-17,2025-11-17,-143
1443,SUB0001444,CUST001319,Cancelled,2026-03-16,NaT,2025-12-04,2025-12-04,-102


Total negative duration records: 42


In [23]:
# Step 17 – Handle Invalid Subscription Dates
invalid_date_mask = (subscription_features["effective_end_date"]< subscription_features["subscription_start_date"])

subscription_features["invalid_date_sequence"] = invalid_date_mask

subscription_features.loc[invalid_date_mask,"subscription_duration_days"] = np.nan

print("Invalid date sequences:",subscription_features["invalid_date_sequence"].sum())

print("Missing subscription durations:",subscription_features["subscription_duration_days"].isna().sum())

print("Negative subscription durations:",(subscription_features["subscription_duration_days"] < 0).sum())

Invalid date sequences: 42
Missing subscription durations: 42
Negative subscription durations: 0


In [24]:
# Step 18 – Create Subscription Status Indicators
subscription_features["is_active_subscription"] = (subscription_features["subscription_status"] == "Active").astype(int)

subscription_features["is_cancelled_subscription"] = (subscription_features["subscription_status"] == "Cancelled").astype(int)

subscription_features["is_plan_changed_subscription"] = (subscription_features["subscription_status"] == "Plan Changed"
).astype(int)

display(
    subscription_features[
        [
            "subscription_status",
            "is_active_subscription",
            "is_cancelled_subscription",
            "is_plan_changed_subscription"
        ]
    ].head(10)
)

display(
    subscription_features[
        [
            "is_active_subscription",
            "is_cancelled_subscription",
            "is_plan_changed_subscription"
        ]
    ].sum()
)

,subscription_status,is_active_subscription,is_cancelled_subscription,is_plan_changed_subscription
0,Active,1,0,0
1,Active,1,0,0
2,Cancelled,0,1,0
3,Active,1,0,0
4,Active,1,0,0
5,Active,1,0,0
6,Cancelled,0,1,0
7,Active,1,0,0
8,Active,1,0,0
9,Active,1,0,0


is_active_subscription          6009
is_cancelled_subscription       1991
is_plan_changed_subscription     875
dtype: int64

In [25]:
# Step 19 – Create Auto-Renewal Feature
subscription_features["is_auto_renew_enabled"] = (
    subscription_features["auto_renew"].astype(int)
)

print("Auto-renewal feature distribution:")
display(
    subscription_features["is_auto_renew_enabled"].value_counts()
)

print("Missing values:",
      subscription_features["is_auto_renew_enabled"].isna().sum())

Auto-renewal feature distribution:


is_auto_renew_enabled
1    7227
0    1648
Name: count, dtype: int64

Missing values: 0


In [26]:
# Step 20 – Calculate Total Subscriptions per Customer
customer_subscription_counts = (
    subscription_features.groupby("customer_id")
    .agg(total_subscriptions=("subscription_id", "nunique"))
    .reset_index()
)

print("Unique customers:", customer_subscription_counts["customer_id"].nunique())

print("Subscription count distribution:")
display(
    customer_subscription_counts["total_subscriptions"].value_counts().sort_index()
)

display(customer_subscription_counts.head(10))

Unique customers: 8000
Subscription count distribution:


total_subscriptions
1    7125
2     875
Name: count, dtype: int64

,customer_id,total_subscriptions
0,CUST000001,1
1,CUST000002,1
2,CUST000003,1
3,CUST000004,1
4,CUST000005,1
5,CUST000006,1
6,CUST000007,1
7,CUST000008,1
8,CUST000009,1
9,CUST000010,1


In [27]:
# Step 21 – Merge Subscription Counts into Customer Features
customer_features = customer_features.merge(
    customer_subscription_counts,
    on="customer_id",
    how="left",
    validate="one_to_one"
)

print("Customer features shape:", customer_features.shape)
print("Missing subscription counts:",
      customer_features["total_subscriptions"].isna().sum())

display(
    customer_features[
        ["customer_id", "age_group", "customer_tenure_days", "total_subscriptions"]
    ].head(10)
)

Customer features shape: (8000, 15)
Missing subscription counts: 0


,customer_id,age_group,customer_tenure_days,total_subscriptions
0,CUST000744,18-24,2584,1
1,CUST001502,25-34,168,1
2,CUST005383,35-44,1109,1
3,CUST004708,25-34,2670,1
4,CUST005877,35-44,209,1
5,CUST003762,25-34,261,1
6,CUST007892,25-34,303,1
7,CUST002317,35-44,2261,1
8,CUST005609,35-44,1274,1
9,CUST000519,25-34,874,2


In [28]:
# Step 22 – Aggregate Subscription Features by Customer
customer_subscription_features = (
    subscription_features.groupby("customer_id")
    .agg(
        active_subscription_count=("is_active_subscription", "sum"),
        cancelled_subscription_count=("is_cancelled_subscription", "sum"),
        plan_changed_subscription_count=("is_plan_changed_subscription", "sum"),
        average_subscription_duration_days=("subscription_duration_days", "mean"),
        average_monthly_price=("monthly_price", "mean"),
        auto_renewal_rate=("is_auto_renew_enabled", "mean")
    )
    .reset_index()
)

print("Customer subscription features shape:",
      customer_subscription_features.shape)

display(customer_subscription_features.head(10))

Customer subscription features shape: (8000, 7)


,customer_id,active_subscription_count,cancelled_subscription_count,plan_changed_subscription_count,average_subscription_duration_days,average_monthly_price,auto_renewal_rate
0,CUST000001,1,0,0,1522.0,22.99,1.0
1,CUST000002,1,0,0,188.0,15.49,1.0
2,CUST000003,0,1,0,2318.0,22.99,1.0
3,CUST000004,1,0,0,2710.0,6.99,0.0
4,CUST000005,1,0,0,706.0,6.99,1.0
5,CUST000006,1,0,0,346.0,22.99,1.0
6,CUST000007,0,1,0,187.0,15.49,1.0
7,CUST000008,1,0,0,1044.0,15.49,1.0
8,CUST000009,1,0,0,2416.0,15.49,1.0
9,CUST000010,1,0,0,1670.0,6.99,1.0


In [29]:
# Step 23 – Validate Aggregated Subscription Features
print("Missing values:")
display(customer_subscription_features.isna().sum())

print("Subscription count reconciliation:")

print(
    "Total subscriptions:",
    customer_subscription_features[
        [
            "active_subscription_count",
            "cancelled_subscription_count",
            "plan_changed_subscription_count"
        ]
    ].sum().sum()
)

print("Unique customers:",
      customer_subscription_features["customer_id"].nunique())

print("Customer feature shape:",
      customer_subscription_features.shape)

Missing values:


customer_id                           0
active_subscription_count             0
cancelled_subscription_count          0
plan_changed_subscription_count       0
average_subscription_duration_days    0
average_monthly_price                 0
auto_renewal_rate                     0
dtype: int64

Subscription count reconciliation:
Total subscriptions: 8875
Unique customers: 8000
Customer feature shape: (8000, 7)


In [30]:
# Step 24 – Merge Subscription Features into Customer Features
customer_features = customer_features.merge(
    customer_subscription_features,
    on="customer_id",
    how="left",
    validate="one_to_one"
)

print("Final customer features shape:", customer_features.shape)

print("Missing values by column:")
display(customer_features.isna().sum())

display(customer_features.head(10))

Final customer features shape: (8000, 21)
Missing values by column:


customer_id                              0
first_name                               0
last_name                                0
age                                      0
gender                                   0
country                                  0
state_or_region                       8000
city                                     0
registration_date                        0
acquisition_channel                      0
customer_segment                         0
preferred_language                       0
age_group                                0
customer_tenure_days                     0
total_subscriptions                      0
active_subscription_count                0
cancelled_subscription_count             0
plan_changed_subscription_count          0
average_subscription_duration_days       0
average_monthly_price                    0
auto_renewal_rate                        0
dtype: int64

,customer_id,first_name,last_name,age,gender,country,state_or_region,city,registration_date,acquisition_channel,...,preferred_language,age_group,customer_tenure_days,total_subscriptions,active_subscription_count,cancelled_subscription_count,plan_changed_subscription_count,average_subscription_duration_days,average_monthly_price,auto_renewal_rate
0,CUST000744,Riya,Santos,21,Female,United Kingdom,NaN,Liverpool,2019-06-03,Organic Search,...,English,18-24,2584,1,1,0,0,2584.0,22.99,0.0
1,CUST001502,Ananya,Hussain,31,Male,Canada,NaN,Montreal,2026-01-13,Referral,...,French,25-34,168,1,1,0,0,168.0,6.99,1.0
2,CUST005383,Zara,Banda,40,Male,United Kingdom,NaN,Leeds,2023-06-17,Paid Social,...,English,35-44,1109,1,1,0,0,1109.0,6.99,1.0
3,CUST004708,Fernando,Okonkwo,27,Male,United Kingdom,NaN,Manchester,2019-03-09,Partner Bundle (Telecom),...,Hindi,25-34,2670,1,0,1,0,2497.0,15.49,0.0
4,CUST005877,Gabriela,Nair,36,Male,Indonesia,NaN,Jakarta,2025-12-03,Email Campaign,...,French,35-44,209,1,1,0,0,209.0,22.99,1.0
5,CUST003762,Elijah,Sheikh,29,Male,Poland,NaN,Warsaw,2025-10-12,App Store,...,English,25-34,261,1,1,0,0,261.0,15.49,0.0
6,CUST007892,Ethan,Lim,31,Female,Japan,NaN,Yokohama,2025-08-31,Direct,...,Hindi,25-34,303,1,1,0,0,303.0,15.49,1.0
7,CUST002317,Ananya,Oliveira,38,Female,India,NaN,Kolkata,2020-04-21,Paid Social,...,English,35-44,2261,1,1,0,0,2261.0,6.99,0.0
8,CUST005609,Isabella,Adeyemi,38,Female,Canada,NaN,Montreal,2023-01-03,App Store,...,Korean,35-44,1274,1,1,0,0,1274.0,15.49,0.0
9,CUST000519,Leo,Nakamura,29,Male,Brazil,NaN,Fortaleza,2024-02-07,Paid Social,...,English,25-34,874,2,1,0,1,437.0,22.99,1.0


In [31]:
# Step 25 – Inspect Viewing Activity Dataset
viewing_activity = cleaned_datasets["viewing_activity"]

print("Dataset shape:", viewing_activity.shape)

print("Columns:")
print(viewing_activity.columns.tolist())

display(viewing_activity.head(10))

Dataset shape: (66422, 9)
Columns:
['viewing_id', 'customer_id', 'content_id', 'viewing_date', 'watch_duration_minutes', 'completion_percentage', 'device_type', 'login_location', 'session_duration_minutes']


,viewing_id,customer_id,content_id,viewing_date,watch_duration_minutes,completion_percentage,device_type,login_location,session_duration_minutes
0,VIEW00000001,CUST000001,CNT00299,2025-08-14,14,100.0,Smart TV,United States,31
1,VIEW00000002,CUST000001,CNT00098,2026-03-07,42,49.0,Smart TV,United States,51
2,VIEW00000003,CUST000001,CNT00187,2026-02-21,50,100.0,Tablet,United States,57
3,VIEW00000004,CUST000001,CNT00126,2026-05-18,71,68.0,Smart TV,United States,83
4,VIEW00000005,CUST000001,CNT00179,2026-05-12,49,69.0,Mobile,United States,64
5,VIEW00000006,CUST000002,CNT00487,2026-04-08,29,60.0,Laptop,United States,45
6,VIEW00000007,CUST000002,CNT00256,2026-05-01,46,45.0,Smart TV,France,57
7,VIEW00000008,CUST000002,CNT00157,2026-01-29,49,47.0,Smart TV,France,56
8,VIEW00000009,CUST000002,CNT00009,2025-12-29,41,79.0,Smart TV,France,49
9,VIEW00000010,CUST000002,CNT00211,2026-04-19,41,97.0,Tablet,France,59


In [32]:
print("Missing values:")
display(viewing_activity.isnull().sum())

print("\nNumerical column statistics:")
display(
    viewing_activity[
        [
            "watch_duration_minutes",
            "completion_percentage",
            "session_duration_minutes"
        ]
    ].describe()
)

print("\nDevice type distribution:")
display(viewing_activity["device_type"].value_counts(dropna=False))

print("\nLogin location distribution:")
display(viewing_activity["login_location"].value_counts(dropna=False).head(15))

print("\nViewing date range:")
print("Earliest:", viewing_activity["viewing_date"].min())
print("Latest:", viewing_activity["viewing_date"].max())

Missing values:


viewing_id                  0
customer_id                 0
content_id                  0
viewing_date                0
watch_duration_minutes      0
completion_percentage       0
device_type                 0
login_location              0
session_duration_minutes    0
dtype: int64


Numerical column statistics:


,watch_duration_minutes,completion_percentage,session_duration_minutes
count,66422.000000,66422.000000,66422.000000
mean,49.433862,70.478486,57.214071
std,40.388162,20.061378,40.733577
min,3.000000,1.000000,3.000000
25%,31.000000,57.000000,38.000000
50%,48.000000,71.000000,55.000000
75%,64.000000,86.000000,73.000000
max,894.000000,100.000000,907.000000



Device type distribution:


device_type
Smart TV      25132
Mobile        19822
Laptop         8333
Tablet         7811
Gaming         3959
Desktop/PC      701
Unknown         664
Name: count, dtype: int64


Login location distribution:


login_location
United States     12297
India             10775
United Kingdom     5318
Canada             3628
Germany            3289
Brazil             3234
France             2655
Mexico             2634
Japan              2477
Indonesia          2109
Philippines        1998
South Korea        1980
Nigeria            1971
Italy              1949
South Africa       1861
Name: count, dtype: int64


Viewing date range:
Earliest: 2025-07-01 00:00:00
Latest: 2026-06-30 00:00:00


In [33]:
display(viewing_activity.isnull().sum())

viewing_id                  0
customer_id                 0
content_id                  0
viewing_date                0
watch_duration_minutes      0
completion_percentage       0
device_type                 0
login_location              0
session_duration_minutes    0
dtype: int64

In [34]:
display(
    viewing_activity[
        [
            "watch_duration_minutes",
            "completion_percentage",
            "session_duration_minutes"
        ]
    ].describe()
)

,watch_duration_minutes,completion_percentage,session_duration_minutes
count,66422.000000,66422.000000,66422.000000
mean,49.433862,70.478486,57.214071
std,40.388162,20.061378,40.733577
min,3.000000,1.000000,3.000000
25%,31.000000,57.000000,38.000000
50%,48.000000,71.000000,55.000000
75%,64.000000,86.000000,73.000000
max,894.000000,100.000000,907.000000


In [35]:
# Step 26 – Remaining checks
print("Device type distribution:")
display(viewing_activity["device_type"].value_counts(dropna=False))

print("\nLogin location distribution:")
display(viewing_activity["login_location"].value_counts(dropna=False).head(15))

print("\nViewing date range:")
print("Earliest:", viewing_activity["viewing_date"].min())
print("Latest:", viewing_activity["viewing_date"].max())




Device type distribution:


device_type
Smart TV      25132
Mobile        19822
Laptop         8333
Tablet         7811
Gaming         3959
Desktop/PC      701
Unknown         664
Name: count, dtype: int64


Login location distribution:


login_location
United States     12297
India             10775
United Kingdom     5318
Canada             3628
Germany            3289
Brazil             3234
France             2655
Mexico             2634
Japan              2477
Indonesia          2109
Philippines        1998
South Korea        1980
Nigeria            1971
Italy              1949
South Africa       1861
Name: count, dtype: int64


Viewing date range:
Earliest: 2025-07-01 00:00:00
Latest: 2026-06-30 00:00:00


In [36]:
display(viewing_activity["device_type"].value_counts(dropna=False))

device_type
Smart TV      25132
Mobile        19822
Laptop         8333
Tablet         7811
Gaming         3959
Desktop/PC      701
Unknown         664
Name: count, dtype: int64

In [37]:
print("Earliest:", viewing_activity["viewing_date"].min())
print("Latest:", viewing_activity["viewing_date"].max())

Earliest: 2025-07-01 00:00:00
Latest: 2026-06-30 00:00:00


In [38]:
# Step 27 – Create customer-level viewing features
customer_viewing_features = (
    viewing_activity.groupby("customer_id")
    .agg(
        total_viewing_sessions=("viewing_id", "nunique"),
        total_watch_duration_minutes=("watch_duration_minutes", "sum"),
        average_watch_duration_minutes=("watch_duration_minutes", "mean"),
        average_completion_percentage=("completion_percentage", "mean"),
        average_session_duration_minutes=("session_duration_minutes", "mean"),
        unique_content_watched=("content_id", "nunique"),
        active_viewing_days=("viewing_date", "nunique"),
        last_viewing_date=("viewing_date", "max")
    )
    .reset_index()
)

customer_viewing_features["days_since_last_viewing"] = (
    snapshot_date - customer_viewing_features["last_viewing_date"]
).dt.days

print("Feature dataset shape:", customer_viewing_features.shape)
display(customer_viewing_features.head())

Feature dataset shape: (7980, 10)


,customer_id,total_viewing_sessions,total_watch_duration_minutes,average_watch_duration_minutes,average_completion_percentage,average_session_duration_minutes,unique_content_watched,active_viewing_days,last_viewing_date,days_since_last_viewing
0,CUST000001,5,226,45.200,77.20,57.200,5,5,2026-05-18,43
1,CUST000002,5,206,41.200,65.60,53.200,5,5,2026-05-01,60
2,CUST000003,2,84,42.000,76.50,49.000,2,2,2026-02-12,138
3,CUST000004,8,471,58.875,63.75,66.375,8,8,2026-06-07,23
4,CUST000005,10,397,39.700,70.00,50.000,10,9,2026-06-24,6


In [39]:
# Step 28 – Validate viewing features
print("Feature dataset shape:", customer_viewing_features.shape)

print("\nUnique customers:")
print(customer_viewing_features["customer_id"].nunique())

print("\nMissing values:")
display(customer_viewing_features.isnull().sum())

print("\nTotal viewing sessions:")
print(customer_viewing_features["total_viewing_sessions"].sum())

print("\nOriginal viewing records:")
print(len(viewing_activity))

print("\nNegative recency values:")
print(
    (customer_viewing_features["days_since_last_viewing"] < 0).sum()
)

Feature dataset shape: (7980, 10)

Unique customers:
7980

Missing values:


customer_id                         0
total_viewing_sessions              0
total_watch_duration_minutes        0
average_watch_duration_minutes      0
average_completion_percentage       0
average_session_duration_minutes    0
unique_content_watched              0
active_viewing_days                 0
last_viewing_date                   0
days_since_last_viewing             0
dtype: int64


Total viewing sessions:
66422

Original viewing records:
66422

Negative recency values:
0


In [40]:
print("Customers without viewing activity:")
print(
    customer_features["customer_id"].nunique()
    - customer_viewing_features["customer_id"].nunique()
)

print("\nNegative recency values:")
print(
    (customer_viewing_features["days_since_last_viewing"] < 0).sum()
)

Customers without viewing activity:
20

Negative recency values:
0


In [41]:
# Step 29 -  Merge viewing features into customer_features
customer_features = customer_features.merge(
    customer_viewing_features,
    on="customer_id",
    how="left",
    validate="one_to_one"
)

customer_features["has_viewing_activity"] = (
    customer_features["last_viewing_date"].notna().astype(int)
)

viewing_metric_columns = [
    "total_viewing_sessions",
    "total_watch_duration_minutes",
    "average_watch_duration_minutes",
    "average_completion_percentage",
    "average_session_duration_minutes",
    "unique_content_watched",
    "active_viewing_days"
]

customer_features[viewing_metric_columns] = (
    customer_features[viewing_metric_columns].fillna(0)
)

print("Updated customer feature shape:", customer_features.shape)
print("\nViewing activity indicator:")
display(customer_features["has_viewing_activity"].value_counts())

print("\nMissing values in viewing features:")
display(customer_features[viewing_metric_columns].isnull().sum())

Updated customer feature shape: (8000, 31)

Viewing activity indicator:


has_viewing_activity
1    7980
0      20
Name: count, dtype: int64


Missing values in viewing features:


total_viewing_sessions              0
total_watch_duration_minutes        0
average_watch_duration_minutes      0
average_completion_percentage       0
average_session_duration_minutes    0
unique_content_watched              0
active_viewing_days                 0
dtype: int64

In [42]:
# Step 30 – Validate the merged customer dataset
print("Customer feature dataset shape:", customer_features.shape)

print("\nUnique customer IDs:")
print(customer_features["customer_id"].nunique())

print("\nDuplicate customer IDs:")
print(customer_features["customer_id"].duplicated().sum())

print("\nMissing last viewing dates:")
print(customer_features["last_viewing_date"].isnull().sum())

print("\nMissing viewing recency:")
print(customer_features["days_since_last_viewing"].isnull().sum())

print("\nTotal viewing sessions:")
print(customer_features["total_viewing_sessions"].sum())

Customer feature dataset shape: (8000, 31)

Unique customer IDs:
8000

Duplicate customer IDs:
0

Missing last viewing dates:
20

Missing viewing recency:
20

Total viewing sessions:
66422.0


In [43]:
# Step 31 – Inspect Payment Dataset
payments = cleaned_datasets["payments"]

print("Dataset shape:", payments.shape)

print("\nColumns:")
print(payments.columns.tolist())

print("\nData types:")
display(payments.dtypes)

print("\nFirst 10 records:")
display(payments.head(10))

Dataset shape: (110735, 8)

Columns:
['payment_id', 'customer_id', 'subscription_id', 'payment_date', 'amount', 'payment_method', 'payment_status', 'failed_payment_reason']

Data types:


payment_id                          str
customer_id                         str
subscription_id                     str
payment_date             datetime64[us]
amount                          float64
payment_method                      str
payment_status                      str
failed_payment_reason               str
dtype: object


First 10 records:


,payment_id,customer_id,subscription_id,payment_date,amount,payment_method,payment_status,failed_payment_reason
0,PAY00000001,CUST000001,SUB0000001,2022-04-30,22.99,Debit Card,Success,NaN
1,PAY00000002,CUST000001,SUB0000001,2022-05-30,22.99,Net Banking,Success,NaN
2,PAY00000003,CUST000001,SUB0000001,2022-06-29,22.99,Debit Card,Success,NaN
3,PAY00000004,CUST000001,SUB0000001,2022-07-29,22.99,Debit Card,Success,NaN
4,PAY00000005,CUST000001,SUB0000001,2022-08-28,22.99,PayPal,Success,NaN
5,PAY00000006,CUST000001,SUB0000001,2022-09-27,11.49,Credit Card,Success,NaN
6,PAY00000007,CUST000001,SUB0000001,2022-10-27,22.99,PayPal,Success,NaN
7,PAY00000008,CUST000001,SUB0000001,2022-11-26,22.99,PayPal,Success,NaN
8,PAY00000009,CUST000001,SUB0000001,2022-12-26,22.99,Gift Card,Success,NaN
9,PAY00000010,CUST000001,SUB0000001,2023-01-25,22.99,Debit Card,Success,NaN


In [44]:
# Step 32 – Inspect Payment Values
print("Missing values:")
display(payments.isnull().sum())

print("\nPayment amount statistics:")
display(payments["amount"].describe())

print("\nPayment status distribution:")
display(payments["payment_status"].value_counts(dropna=False))

print("\nPayment method distribution:")
display(payments["payment_method"].value_counts(dropna=False))

print("\nFailed payment reasons:")
display(payments["failed_payment_reason"].value_counts(dropna=False))

print("\nPayment date range:")
print("Earliest:", payments["payment_date"].min())
print("Latest:", payments["payment_date"].max())

Missing values:


payment_id                    0
customer_id                   0
subscription_id            1045
payment_date                  0
amount                        0
payment_method                0
payment_status                0
failed_payment_reason    103529
dtype: int64


Payment amount statistics:


count    110735.000000
mean         13.750196
std           9.510979
min           0.000000
25%           6.990000
50%          15.490000
75%          15.490000
max         252.890000
Name: amount, dtype: float64


Payment status distribution:


payment_status
Success    103529
Failed       7206
Name: count, dtype: int64


Payment method distribution:


payment_method
Credit Card      33249
Debit Card       19751
UPI              17730
PayPal           13375
Net Banking      11344
Mobile Wallet     8697
Gift Card         6589
Name: count, dtype: int64


Failed payment reasons:


failed_payment_reason
NaN                                  103529
Bank declined                          1278
Card expired                           1211
Incorrect card details                 1198
Insufficient funds                     1189
Card reported lost/stolen              1180
International transaction blocked      1150
Name: count, dtype: int64


Payment date range:
Earliest: 2019-01-01 00:00:00
Latest: 2026-05-31 00:00:00


In [45]:
# remaining checks
print("Failed payment reasons:")
display(
    payments.loc[
        payments["payment_status"] == "Failed",
        "failed_payment_reason"
    ].value_counts(dropna=False)
)

print("\nPayment date range:")
print("Earliest:", payments["payment_date"].min())
print("Latest:", payments["payment_date"].max())

Failed payment reasons:


failed_payment_reason
Bank declined                        1278
Card expired                         1211
Incorrect card details               1198
Insufficient funds                   1189
Card reported lost/stolen            1180
International transaction blocked    1150
Name: count, dtype: int64


Payment date range:
Earliest: 2019-01-01 00:00:00
Latest: 2026-05-31 00:00:00


In [46]:
print("Earliest:", payments["payment_date"].min())
print("Latest:", payments["payment_date"].max())

Earliest: 2019-01-01 00:00:00
Latest: 2026-05-31 00:00:00


In [47]:
# Step 33 – Payment Feature Engineering
payment_features = payments.copy()

payment_features["is_successful_payment"] = (
    payment_features["payment_status"] == "Success"
).astype(int)

payment_features["is_failed_payment"] = (
    payment_features["payment_status"] == "Failed"
).astype(int)

payment_features["successful_payment_amount"] = (
    payment_features["amount"].where(
        payment_features["is_successful_payment"] == 1
    )
)

payment_features["successful_payment_date"] = (
    payment_features["payment_date"].where(
        payment_features["is_successful_payment"] == 1
    )
)

customer_payment_features = (
    payment_features.groupby("customer_id")
    .agg(
        total_payment_transactions=("payment_id", "nunique"),
        successful_payment_count=("is_successful_payment", "sum"),
        failed_payment_count=("is_failed_payment", "sum"),
        total_amount_paid=("successful_payment_amount", "sum"),
        average_successful_payment_amount=("successful_payment_amount", "mean"),
        payment_success_rate=("is_successful_payment", "mean"),
        unique_payment_methods=("payment_method", "nunique"),
        most_used_payment_method=(
            "payment_method",
            lambda x: x.mode().iloc[0]
        ),
        last_payment_date=("payment_date", "max"),
        last_successful_payment_date=("successful_payment_date", "max")
    )
    .reset_index()
)

customer_payment_features["days_since_last_successful_payment"] = (
    snapshot_date
    - customer_payment_features["last_successful_payment_date"]
).dt.days

print("Payment feature dataset shape:", customer_payment_features.shape)

display(customer_payment_features.head())

Payment feature dataset shape: (8000, 12)


,customer_id,total_payment_transactions,successful_payment_count,failed_payment_count,total_amount_paid,average_successful_payment_amount,payment_success_rate,unique_payment_methods,most_used_payment_method,last_payment_date,last_successful_payment_date,days_since_last_successful_payment
0,CUST000001,15,15,0,333.35,22.223333,1.000000,5,Debit Card,2023-06-24,2023-06-24,1102.0
1,CUST000002,6,6,0,92.94,15.490000,1.000000,3,Debit Card,2026-05-23,2026-05-23,38.0
2,CUST000003,15,15,0,344.85,22.990000,1.000000,7,Credit Card,2021-02-07,2021-02-07,1969.0
3,CUST000004,15,14,1,97.86,6.990000,0.933333,7,Debit Card,2020-03-23,2020-03-23,2290.0
4,CUST000005,15,14,1,97.86,6.990000,0.933333,7,Credit Card,2025-09-17,2025-09-17,286.0


In [48]:
# Step 34 – Validate Payment Features
print("Feature dataset shape:", customer_payment_features.shape)

print("\nUnique customers:")
print(customer_payment_features["customer_id"].nunique())

print("\nMissing values:")
display(customer_payment_features.isnull().sum())

print("\nTotal payment transactions:")
print(customer_payment_features["total_payment_transactions"].sum())

print("\nSuccessful payments:")
print(customer_payment_features["successful_payment_count"].sum())

print("\nFailed payments:")
print(customer_payment_features["failed_payment_count"].sum())

print("\nCustomers with no successful payments:")
print(
    (customer_payment_features["successful_payment_count"] == 0).sum()
)

print("\nPayment success rate range:")
print(
    customer_payment_features["payment_success_rate"].min(),
    customer_payment_features["payment_success_rate"].max()
)

Feature dataset shape: (8000, 12)

Unique customers:
8000

Missing values:


customer_id                            0
total_payment_transactions             0
successful_payment_count               0
failed_payment_count                   0
total_amount_paid                      0
average_successful_payment_amount     10
payment_success_rate                   0
unique_payment_methods                 0
most_used_payment_method               0
last_payment_date                      0
last_successful_payment_date          10
days_since_last_successful_payment    10
dtype: int64


Total payment transactions:
110735

Successful payments:
103529

Failed payments:
7206

Customers with no successful payments:
10

Payment success rate range:
0.0 1.0


In [49]:
# Step 35 – Merge Payment Features
customer_payment_features["has_successful_payment"] = (
    customer_payment_features["successful_payment_count"] > 0
).astype(int)

customer_payment_features["average_successful_payment_amount"] = (
    customer_payment_features["average_successful_payment_amount"].fillna(0)
)

customer_features = customer_features.merge(
    customer_payment_features,
    on="customer_id",
    how="left",
    validate="one_to_one"
)

print("Updated customer feature shape:", customer_features.shape)

print("\nSuccessful payment indicator:")
display(customer_features["has_successful_payment"].value_counts())

print("\nMissing payment metrics:")
display(customer_features.isnull().sum().tail(12))

Updated customer feature shape: (8000, 43)

Successful payment indicator:


has_successful_payment
1    7990
0      10
Name: count, dtype: int64


Missing payment metrics:


total_payment_transactions             0
successful_payment_count               0
failed_payment_count                   0
total_amount_paid                      0
average_successful_payment_amount      0
payment_success_rate                   0
unique_payment_methods                 0
most_used_payment_method               0
last_payment_date                      0
last_successful_payment_date          10
days_since_last_successful_payment    10
has_successful_payment                 0
dtype: int64

In [50]:
# Step 36 – Payment Integration Validation
print("Customer dataset shape:", customer_features.shape)

print("Unique customers:", customer_features["customer_id"].nunique())

print("Duplicate customer IDs:", customer_features["customer_id"].duplicated().sum())

print("Total payment transactions:", customer_features["total_payment_transactions"].sum())

print("Total successful payments:", customer_features["successful_payment_count"].sum())

print("Total failed payments:", customer_features["failed_payment_count"].sum())

Customer dataset shape: (8000, 43)
Unique customers: 8000
Duplicate customer IDs: 0
Total payment transactions: 110735
Total successful payments: 103529
Total failed payments: 7206


In [51]:
# Step 37 – Support Ticket Feature Engineering
support_tickets = cleaned_datasets["support_tickets"].copy()

print("Support tickets shape:", support_tickets.shape)

print("\nColumns:")
print(support_tickets.columns.tolist())

print("\nData types:")
print(support_tickets.dtypes)

display(support_tickets.head())

Support tickets shape: (6120, 11)

Columns:
['ticket_id', 'customer_id', 'ticket_date', 'issue_category', 'issue_subcategory', 'priority', 'ticket_status', 'resolution_time_hours', 'customer_satisfaction_score', 'ticket_description', 'support_agent_id']

Data types:
ticket_id                                 str
customer_id                               str
ticket_date                    datetime64[us]
issue_category                            str
issue_subcategory                         str
priority                                  str
ticket_status                             str
resolution_time_hours                 float64
customer_satisfaction_score           float64
ticket_description                        str
support_agent_id                          str
dtype: object


,ticket_id,customer_id,ticket_date,issue_category,issue_subcategory,priority,ticket_status,resolution_time_hours,customer_satisfaction_score,ticket_description,support_agent_id
0,TCK0000001,CUST000001,2024-12-05,Subscription,Auto-renew complaint,Medium,Closed,15.8,2.0,I was auto renewed without any reminder notifi...,AGT0050
1,TCK0000002,CUST000001,2025-02-06,Account Access,Login failure,Medium,Closed,6.0,3.0,I cannot log into my account from my phone eve...,AGT0120
2,TCK0000003,CUST000002,2026-06-15,Billing & Payments,Failed payment,Medium,Resolved,6.4,3.0,My payment failed twice this month even though...,AGT0012
3,TCK0000004,CUST000003,2021-03-05,Account Access,Account locked,Low,Escalated,NaN,NaN,My account got locked after multiple login att...,AGT0101
4,TCK0000005,CUST000007,2026-02-16,Account Access,Account locked,Medium,Resolved,2.4,3.0,My account got locked after multiple login att...,AGT0109


In [52]:
# Step 37.1 – Inspect support ticket values
print("Missing values:")
print(support_tickets.isnull().sum())

print("\nTicket status distribution:")
print(support_tickets["ticket_status"].value_counts(dropna=False))

print("\nPriority distribution:")
print(support_tickets["priority"].value_counts(dropna=False))

print("\nIssue category distribution:")
print(support_tickets["issue_category"].value_counts(dropna=False))

print("\nCustomer satisfaction statistics:")
print(support_tickets["customer_satisfaction_score"].describe())

print("\nResolution time statistics:")
print(support_tickets["resolution_time_hours"].describe())

Missing values:
ticket_id                         0
customer_id                       0
ticket_date                      61
issue_category                    0
issue_subcategory                 0
priority                          0
ticket_status                     0
resolution_time_hours          1528
customer_satisfaction_score    1528
ticket_description                0
support_agent_id                  0
dtype: int64

Ticket status distribution:
ticket_status
Resolved     3390
Closed       1202
Pending       888
Escalated     640
Name: count, dtype: int64

Priority distribution:
priority
Low         2204
Medium      2087
High        1328
Critical     501
Name: count, dtype: int64

Issue category distribution:
issue_category
Billing & Payments    1709
Streaming Quality     1323
Account Access         998
Subscription           881
Device & App           738
Content                471
Name: count, dtype: int64

Customer satisfaction statistics:
count    4592.000000
mean        3.2584

In [53]:
# Step 37.2 – Validate metrics by ticket status
support_tickets.groupby("ticket_status").agg(
    ticket_count=("ticket_id", "nunique"),
    resolution_records=("resolution_time_hours", "count"),
    average_resolution_hours=("resolution_time_hours", "mean"),
    satisfaction_records=("customer_satisfaction_score", "count"),
    average_satisfaction=("customer_satisfaction_score", "mean")
)

,ticket_count,resolution_records,average_resolution_hours,satisfaction_records,average_satisfaction
ticket_status,,,,,
Closed,1202,1202,17.846506,1202,3.250416
Escalated,640,0,NaN,0,NaN
Pending,888,0,NaN,0,NaN
Resolved,3390,3390,18.052950,3390,3.261357


In [54]:
support_features = support_tickets.copy()

support_features["is_resolved"] = (support_features["ticket_status"] == "Resolved").astype(int)

support_features["is_closed"] = (support_features["ticket_status"] == "Closed").astype(int)

support_features["is_pending"] = (support_features["ticket_status"] == "Pending").astype(int)

support_features["is_escalated"] = (support_features["ticket_status"] == "Escalated").astype(int)

customer_support_features = (
    support_features.groupby("customer_id")
    .agg(
        total_support_tickets=("ticket_id", "nunique"),
        resolved_ticket_count=("is_resolved", "sum"),
        closed_ticket_count=("is_closed", "sum"),
        pending_ticket_count=("is_pending", "sum"),
        escalated_ticket_count=("is_escalated", "sum"),
        average_resolution_time_hours=("resolution_time_hours", "mean"),
        avg_customer_satisfaction=("customer_satisfaction_score", "mean"),
        unique_issue_categories=("issue_category", "nunique")
    )
    .reset_index()
)

print("Customer support features shape:", customer_support_features.shape)
display(customer_support_features.head())

Customer support features shape: (4408, 9)


,customer_id,total_support_tickets,resolved_ticket_count,closed_ticket_count,pending_ticket_count,escalated_ticket_count,average_resolution_time_hours,avg_customer_satisfaction,unique_issue_categories
0,CUST000001,2,0,2,0,0,10.9,2.5,2
1,CUST000002,1,1,0,0,0,6.4,3.0,1
2,CUST000003,1,0,0,0,1,NaN,NaN,1
3,CUST000007,1,1,0,0,0,2.4,3.0,1
4,CUST000010,1,1,0,0,0,58.6,5.0,1


In [55]:
# Step 37.4 – Validate support features
print("Support feature dataset shape:", customer_support_features.shape)

print("Unique customers:", customer_support_features["customer_id"].nunique())

print("Duplicate customer IDs:", customer_support_features["customer_id"].duplicated().sum())

print("Total support tickets:", customer_support_features["total_support_tickets"].sum())

print("Resolved tickets:", customer_support_features["resolved_ticket_count"].sum())

print("Closed tickets:", customer_support_features["closed_ticket_count"].sum())

print("Pending tickets:", customer_support_features["pending_ticket_count"].sum())

print("Escalated tickets:", customer_support_features["escalated_ticket_count"].sum())

print("Customers without resolution metrics:", customer_support_features["average_resolution_time_hours"].isna().sum())

print("Customers without satisfaction metrics:", customer_support_features["avg_customer_satisfaction"].isna().sum())

Support feature dataset shape: (4408, 9)
Unique customers: 4408
Duplicate customer IDs: 0
Total support tickets: 6120
Resolved tickets: 3390
Closed tickets: 1202
Pending tickets: 888
Escalated tickets: 640
Customers without resolution metrics: 868
Customers without satisfaction metrics: 868


In [56]:
# Step 37.5 – Merge support features into customer dataset
customer_features = customer_features.merge(
    customer_support_features,
    on="customer_id",
    how="left",
    validate="one_to_one"
)

support_count_columns = [
    "total_support_tickets",
    "resolved_ticket_count",
    "closed_ticket_count",
    "pending_ticket_count",
    "escalated_ticket_count",
    "unique_issue_categories"
]

customer_features[support_count_columns] = (
    customer_features[support_count_columns].fillna(0)
)

customer_features["has_support_tickets"] = (
    customer_features["total_support_tickets"] > 0
).astype(int)

customer_features["has_completed_support_ticket"] = (
    customer_features["resolved_ticket_count"]
    + customer_features["closed_ticket_count"] > 0
).astype(int)

print("Customer dataset shape:", customer_features.shape)

print("\nCustomers with support tickets:")
print(customer_features["has_support_tickets"].value_counts())

print("\nCustomers with completed support tickets:")
print(customer_features["has_completed_support_ticket"].value_counts())

print("\nMissing support metrics:")
print(customer_features[
    ["average_resolution_time_hours", "avg_customer_satisfaction"]
].isnull().sum())

Customer dataset shape: (8000, 53)

Customers with support tickets:
has_support_tickets
1    4408
0    3592
Name: count, dtype: int64

Customers with completed support tickets:
has_completed_support_ticket
0    4460
1    3540
Name: count, dtype: int64

Missing support metrics:
average_resolution_time_hours    4460
avg_customer_satisfaction        4460
dtype: int64


In [57]:
# Step 38 – Customer Feedback Feature Engineering
customer_feedback = cleaned_datasets["customer_feedback"].copy()

print("Customer feedback shape:", customer_feedback.shape)

print("\nColumns:")
print(customer_feedback.columns.tolist())

print("\nData types:")
print(customer_feedback.dtypes)

display(customer_feedback.head())

Customer feedback shape: (5046, 6)

Columns:
['feedback_id', 'customer_id', 'feedback_date', 'rating', 'feedback_text', 'sentiment_label']

Data types:
feedback_id                   str
customer_id                   str
feedback_date      datetime64[us]
rating                      int64
feedback_text                 str
sentiment_label               str
dtype: object


,feedback_id,customer_id,feedback_date,rating,feedback_text,sentiment_label
0,FB0000001,CUST000001,2025-12-25,1,"Not enough new content being added, feels like...",Negative
1,FB0000002,CUST000001,2022-08-12,5,"Great value for the price, my whole family use...",Positive
2,FB0000003,CUST000002,2026-04-11,1,Customer support took too long to respond to m...,Negative
3,FB0000004,CUST000003,2024-06-03,3,"Average experience overall, some months are be...",Neutral
4,FB0000005,CUST000008,2024-05-30,3,The service is decent but the content library ...,Neutral


In [58]:
# Step 38.2 – Inspect feedback values
print("Missing values:")
print(customer_feedback.isnull().sum())

print("\nRating distribution:")
print(customer_feedback["rating"].value_counts().sort_index())

print("\nRating statistics:")
print(customer_feedback["rating"].describe())

print("\nSentiment distribution:")
print(customer_feedback["sentiment_label"].value_counts(dropna=False))

print("\nUnique sentiment labels:")
print(customer_feedback["sentiment_label"].unique())

print("\nFeedback date range:")
print(customer_feedback["feedback_date"].min())
print(customer_feedback["feedback_date"].max())

Missing values:
feedback_id        0
customer_id        0
feedback_date      0
rating             0
feedback_text      0
sentiment_label    0
dtype: int64

Rating distribution:
rating
1     725
2     711
3    1599
4    1032
5     979
Name: count, dtype: int64

Rating statistics:
count    5046.000000
mean        3.164289
std         1.292105
min         1.000000
25%         2.000000
50%         3.000000
75%         4.000000
max         5.000000
Name: rating, dtype: float64

Sentiment distribution:
sentiment_label
Positive    1978
Neutral     1606
Negative    1462
Name: count, dtype: int64

Unique sentiment labels:
<ArrowStringArray>
['Negative', 'Positive', 'Neutral']
Length: 3, dtype: str

Feedback date range:
2019-03-06 00:00:00
2026-06-30 00:00:00


In [59]:
# Step 38.3 – Check sentiment distribution
print(customer_feedback["sentiment_label"].value_counts())

print("\nUnique sentiment labels:")
print(customer_feedback["sentiment_label"].unique())

sentiment_label
Positive    1978
Neutral     1606
Negative    1462
Name: count, dtype: int64

Unique sentiment labels:
<ArrowStringArray>
['Negative', 'Positive', 'Neutral']
Length: 3, dtype: str


In [60]:
# Step 38.4 – Create customer-level feedback features
customer_feedback_features = customer_feedback.copy()

customer_feedback_features["is_positive"] = (
    customer_feedback_features["sentiment_label"] == "Positive"
).astype(int)

customer_feedback_features["is_neutral"] = (
    customer_feedback_features["sentiment_label"] == "Neutral"
).astype(int)

customer_feedback_features["is_negative"] = (
    customer_feedback_features["sentiment_label"] == "Negative"
).astype(int)

customer_feedback_features = (
    customer_feedback_features.groupby("customer_id")
    .agg(
        total_feedback_count=("feedback_id", "nunique"),
        average_feedback_rating=("rating", "mean"),
        positive_feedback_count=("is_positive", "sum"),
        neutral_feedback_count=("is_neutral", "sum"),
        negative_feedback_count=("is_negative", "sum"),
        unique_sentiment_categories=("sentiment_label", "nunique"),
        last_feedback_date=("feedback_date", "max")
    )
    .reset_index()
)

customer_feedback_features["positive_feedback_rate"] = (
    customer_feedback_features["positive_feedback_count"]
    / customer_feedback_features["total_feedback_count"]
)

customer_feedback_features["negative_feedback_rate"] = (
    customer_feedback_features["negative_feedback_count"]
    / customer_feedback_features["total_feedback_count"]
)

customer_feedback_features["days_since_last_feedback"] = (
    snapshot_date - customer_feedback_features["last_feedback_date"]
).dt.days

print("Customer feedback features shape:", customer_feedback_features.shape)

display(customer_feedback_features.head())

Customer feedback features shape: (3561, 11)


,customer_id,total_feedback_count,average_feedback_rating,positive_feedback_count,neutral_feedback_count,negative_feedback_count,unique_sentiment_categories,last_feedback_date,positive_feedback_rate,negative_feedback_rate,days_since_last_feedback
0,CUST000001,2,3.0,1,0,1,2,2025-12-25,0.5,0.5,187
1,CUST000002,1,1.0,0,0,1,1,2026-04-11,0.0,1.0,80
2,CUST000003,1,3.0,0,1,0,1,2024-06-03,0.0,0.0,757
3,CUST000008,1,3.0,0,1,0,1,2024-05-30,0.0,0.0,761
4,CUST000009,1,4.0,1,0,0,1,2026-03-14,1.0,0.0,108


In [61]:
# Step 38.5 – Validate feedback features
print("Feedback feature dataset shape:", customer_feedback_features.shape)

print("Unique customers:", customer_feedback_features["customer_id"].nunique())

print("Duplicate customer IDs:", customer_feedback_features["customer_id"].duplicated().sum())

print("Total feedback records:", customer_feedback_features["total_feedback_count"].sum())

print("Positive feedback:", customer_feedback_features["positive_feedback_count"].sum())

print("Neutral feedback:", customer_feedback_features["neutral_feedback_count"].sum())

print("Negative feedback:", customer_feedback_features["negative_feedback_count"].sum())

print("Missing average ratings:", customer_feedback_features["average_feedback_rating"].isna().sum())

print("Negative feedback recency values:", (customer_feedback_features["days_since_last_feedback"] < 0).sum())

Feedback feature dataset shape: (3561, 11)
Unique customers: 3561
Duplicate customer IDs: 0
Total feedback records: 5046
Positive feedback: 1978
Neutral feedback: 1606
Negative feedback: 1462
Missing average ratings: 0
Negative feedback recency values: 0


In [62]:
# Step 38.6 – Merge feedback features into customer dataset
customer_features = customer_features.merge(
    customer_feedback_features,
    on="customer_id",
    how="left",
    validate="one_to_one"
)

feedback_count_columns = [
    "total_feedback_count",
    "positive_feedback_count",
    "neutral_feedback_count",
    "negative_feedback_count",
    "unique_sentiment_categories"
]

feedback_rate_columns = [
    "positive_feedback_rate",
    "negative_feedback_rate"
]

customer_features[feedback_count_columns + feedback_rate_columns] = (
    customer_features[feedback_count_columns + feedback_rate_columns].fillna(0)
)

customer_features["has_feedback"] = (
    customer_features["total_feedback_count"] > 0
).astype(int)

print("Customer dataset shape:", customer_features.shape)

print("\nCustomers with feedback:")
print(customer_features["has_feedback"].value_counts())

print("\nMissing feedback metrics:")
print(customer_features[
    [
        "average_feedback_rating",
        "last_feedback_date",
        "days_since_last_feedback"
    ]
].isnull().sum())

Customer dataset shape: (8000, 64)

Customers with feedback:
has_feedback
0    4439
1    3561
Name: count, dtype: int64

Missing feedback metrics:


average_feedback_rating     4439
last_feedback_date          4439
days_since_last_feedback    4439
dtype: int64


In [63]:
# Step 39 – Content Preference Feature Engineering
# Step 39.1 – Inspect content dataset
content = cleaned_datasets["content"].copy()

print("Content dataset shape:", content.shape)

print("\nColumns:")
print(content.columns.tolist())

print("\nData types:")
print(content.dtypes)

display(content.head())

Content dataset shape: (500, 9)

Columns:
['content_id', 'title', 'content_type', 'genre', 'release_year', 'duration_minutes', 'content_language', 'production_country', 'maturity_rating']

Data types:
content_id              str
title                   str
content_type            str
genre                   str
release_year          int64
duration_minutes      int64
content_language        str
production_country      str
maturity_rating         str
dtype: object


,content_id,title,content_type,genre,release_year,duration_minutes,content_language,production_country,maturity_rating
0,CNT00001,Golden Horizon,TV Show,Horror,2021,51,English,India,TV-14
1,CNT00002,Restless Kingdom,TV Show,Comedy,2004,39,English,United States,TV-MA
2,CNT00003,Silent Symphony,Movie,Family,2016,117,English,India,NR
3,CNT00004,Silent Legacy,TV Show,Stand-Up Comedy,2004,72,Indonesian,India,PG-13
4,CNT00005,Golden Kingdom,Movie,Reality,2017,100,English,Philippines,G


In [64]:
# Step 39.2 – Inspect content categories
print("Missing values:")
print(content.isnull().sum())

print("\nGenre distribution:")
print(content["genre"].value_counts())

print("\nContent type distribution:")
print(content["content_type"].value_counts())

print("\nContent language distribution:")
print(content["content_language"].value_counts())

print("\nUnique content IDs:", content["content_id"].nunique())

print("Duplicate content IDs:", content["content_id"].duplicated().sum())

Missing values:
content_id            0
title                 0
content_type          0
genre                 0
release_year          0
duration_minutes      0
content_language      0
production_country    0
maturity_rating       0
dtype: int64

Genre distribution:
genre
Anime              42
Horror             40
Comedy             39
Family             36
Sci-Fi             34
Stand-Up Comedy    33
Thriller           33
Documentary        32
Reality            29
Mystery            29
Sports             29
Fantasy            29
Action             27
Romance            25
Crime              22
Drama              21
Name: count, dtype: int64

Content type distribution:
content_type
Movie             216
TV Show           144
Limited Series     59
Documentary        55
Stand-Up           26
Name: count, dtype: int64

Content language distribution:
content_language
English       170
Hindi          87
Spanish        43
Portuguese     35
Korean         33
French         29
German         2

In [65]:
# Step 39.3 – Join viewing activity with content information
viewing_content = viewing_activity.merge(
    content,
    on="content_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

print("Viewing content shape:", viewing_content.shape)

print("\nContent matching results:")
print(viewing_content["_merge"].value_counts())

print("\nGenre distribution:")
print(viewing_content["genre"].value_counts())

print("\nContent type distribution:")
print(viewing_content["content_type"].value_counts())

print("\nUnique content IDs:", content["content_id"].nunique())

print("Duplicate content IDs:", content["content_id"].duplicated().sum())

display(viewing_content.head())

Viewing content shape: (66422, 18)

Content matching results:
_merge
both          66422
left_only         0
right_only        0
Name: count, dtype: int64

Genre distribution:
genre
Anime              5556
Horror             5325
Comedy             5202
Family             4820
Sci-Fi             4446
Thriller           4381
Stand-Up Comedy    4273
Documentary        4224
Reality            3896
Fantasy            3886
Sports             3849
Mystery            3829
Action             3565
Romance            3327
Crime              2990
Drama              2853
Name: count, dtype: int64

Content type distribution:
content_type
Movie             28754
TV Show           19012
Limited Series     7888
Documentary        7436
Stand-Up           3332
Name: count, dtype: int64

Unique content IDs: 500
Duplicate content IDs: 0


,viewing_id,customer_id,content_id,viewing_date,watch_duration_minutes,completion_percentage,device_type,login_location,session_duration_minutes,title,content_type,genre,release_year,duration_minutes,content_language,production_country,maturity_rating,_merge
0,VIEW00000001,CUST000001,CNT00299,2025-08-14,14,100.0,Smart TV,United States,31,Northern Circuit,Movie,Comedy,2012,121,English,India,NR,both
1,VIEW00000002,CUST000001,CNT00098,2026-03-07,42,49.0,Smart TV,United States,51,Crimson Signal,Movie,Thriller,2003,110,Spanish,United States,PG,both
2,VIEW00000003,CUST000001,CNT00187,2026-02-21,50,100.0,Tablet,United States,57,Broken Circuit,TV Show,Action,2019,51,Hindi,Philippines,TV-14,both
3,VIEW00000004,CUST000001,CNT00126,2026-05-18,71,68.0,Smart TV,United States,83,Quiet Horizon,Stand-Up,Anime,2016,85,Korean,India,NR,both
4,VIEW00000005,CUST000001,CNT00179,2026-05-12,49,69.0,Mobile,United States,64,Distant Legacy,Movie,Romance,2000,99,Indonesian,Philippines,PG,both


In [66]:
# Step 39.4 – Create customer content preference features
viewing_content = viewing_content.drop(columns=["_merge"])

viewing_content["is_movie"] = (viewing_content["content_type"] == "Movie").astype(int)

viewing_content["is_tv_show"] = (viewing_content["content_type"] == "TV Show").astype(int)

customer_content_features = (
    viewing_content.groupby("customer_id")
    .agg(
        total_content_sessions=("viewing_id", "nunique"),
        unique_genres_watched=("genre", "nunique"),
        movie_viewing_count=("is_movie", "sum"),
        tv_show_viewing_count=("is_tv_show", "sum"),
        preferred_content_type=(
            "content_type",
            lambda x: x.mode().iloc[0]
        ),
        preferred_content_language=(
            "content_language",
            lambda x: x.mode().iloc[0]
        ),
        unique_content_languages_watched=("content_language", "nunique"),
        unique_maturity_ratings_watched=("maturity_rating", "nunique")
    )
    .reset_index()
)

genre_counts = (
    viewing_content.groupby(["customer_id", "genre"])
    .size()
    .reset_index(name="genre_view_count")
)

most_watched_genre = genre_counts.loc[
    genre_counts.groupby("customer_id")["genre_view_count"].idxmax()
].copy()

most_watched_genre = most_watched_genre.rename(
    columns={
        "genre": "most_watched_genre",
        "genre_view_count": "most_watched_genre_count"
    }
)

customer_content_features = customer_content_features.merge(
    most_watched_genre[
        ["customer_id", "most_watched_genre", "most_watched_genre_count"]
    ],
    on="customer_id",
    how="left",
    validate="one_to_one"
)

customer_content_features["preferred_genre_share"] = (
    customer_content_features["most_watched_genre_count"]
    / customer_content_features["total_content_sessions"]
)

print("Customer content features shape:", customer_content_features.shape)

display(customer_content_features.head())

Customer content features shape: (7980, 12)


,customer_id,total_content_sessions,unique_genres_watched,movie_viewing_count,tv_show_viewing_count,preferred_content_type,preferred_content_language,unique_content_languages_watched,unique_maturity_ratings_watched,most_watched_genre,most_watched_genre_count,preferred_genre_share
0,CUST000001,5,5,3,1,Movie,English,5,3,Action,1,0.20
1,CUST000002,5,5,3,2,Movie,English,3,4,Anime,1,0.20
2,CUST000003,2,2,2,0,Movie,English,2,2,Action,1,0.50
3,CUST000004,8,7,5,2,Movie,English,4,5,Fantasy,2,0.25
4,CUST000005,10,8,4,4,Movie,Hindi,6,6,Anime,2,0.20


In [67]:
# Step 39.5 – Validate customer content features
print("Shape:", customer_content_features.shape)
print("Unique customers:", customer_content_features["customer_id"].nunique())
print("Duplicate customer IDs:", customer_content_features["customer_id"].duplicated().sum())

print("\nTotal content sessions:", customer_content_features["total_content_sessions"].sum())
print(
    "Total movie + TV show sessions:",
    customer_content_features["movie_viewing_count"].sum()
    + customer_content_features["tv_show_viewing_count"].sum()
)

print("\nMissing values:")
print(customer_content_features.isna().sum())

print("\nPreferred language empty strings:")
print(
    (customer_content_features["preferred_content_language"] == "").sum()
)

print("\nPreferred genre share range:")
print(
    customer_content_features["preferred_genre_share"].min(),
    customer_content_features["preferred_genre_share"].max()
)

Shape: (7980, 12)
Unique customers: 7980
Duplicate customer IDs: 0

Total content sessions: 66422
Total movie + TV show sessions: 47766

Missing values:
customer_id                         0
total_content_sessions              0
unique_genres_watched               0
movie_viewing_count                 0
tv_show_viewing_count               0
preferred_content_type              0
preferred_content_language          0
unique_content_languages_watched    0
unique_maturity_ratings_watched     0
most_watched_genre                  0
most_watched_genre_count            0
preferred_genre_share               0
dtype: int64

Preferred language empty strings:
0

Preferred genre share range:
0.07692307692307693 1.0


In [68]:
# Step 39.5.1 – Check content types
print(viewing_content["content_type"].value_counts(dropna=False))

print("\nMovie flag total:", viewing_content["is_movie"].sum())
print("TV show flag total:", viewing_content["is_tv_show"].sum())
print("Other content type sessions:")
print(
    viewing_content.loc[
        ~viewing_content["content_type"].isin(["Movie", "TV Show"]),
        "content_type"
    ].value_counts(dropna=False)
)

content_type
Movie             28754
TV Show           19012
Limited Series     7888
Documentary        7436
Stand-Up           3332
Name: count, dtype: int64

Movie flag total: 28754
TV show flag total: 19012
Other content type sessions:
content_type
Limited Series    7888
Documentary       7436
Stand-Up          3332
Name: count, dtype: int64


In [69]:
# Step 39.5.2 – Add remaining content-type features
viewing_content["is_limited_series"] = (viewing_content["content_type"] == "Limited Series").astype(int)

viewing_content["is_documentary"] = (viewing_content["content_type"] == "Documentary").astype(int)

viewing_content["is_stand_up"] = (viewing_content["content_type"] == "Stand-Up").astype(int)

additional_content_features = (
    viewing_content.groupby("customer_id")
    .agg(
        limited_series_viewing_count=("is_limited_series", "sum"),
        documentary_viewing_count=("is_documentary", "sum"),
        stand_up_viewing_count=("is_stand_up", "sum")
    )
    .reset_index()
)

customer_content_features = customer_content_features.merge(
    additional_content_features,
    on="customer_id",
    how="left",
    validate="one_to_one"
)

print("Updated shape:", customer_content_features.shape)

print("Total classified sessions:",customer_content_features[
        [
            "movie_viewing_count",
            "tv_show_viewing_count",
            "limited_series_viewing_count",
            "documentary_viewing_count",
            "stand_up_viewing_count"
        ]
    ].sum().sum()
)

Updated shape: (7980, 15)
Total classified sessions: 66422


In [70]:
# Step 39.5.3 – Validate all five content categories
content_type_columns = [
    "movie_viewing_count",
    "tv_show_viewing_count",
    "limited_series_viewing_count",
    "documentary_viewing_count",
    "stand_up_viewing_count"
]

print("Customer IDs:", customer_content_features["customer_id"].nunique())
print("Duplicate IDs:", customer_content_features["customer_id"].duplicated().sum())

print("\nFeature totals:")
print(customer_content_features[content_type_columns].sum())

print("\nTotal classified sessions:")
print(customer_content_features[content_type_columns].sum().sum())

print("\nNegative counts:")
print((customer_content_features[content_type_columns] < 0).sum())

Customer IDs: 7980
Duplicate IDs: 0

Feature totals:
movie_viewing_count             28754
tv_show_viewing_count           19012
limited_series_viewing_count     7888
documentary_viewing_count        7436
stand_up_viewing_count           3332
dtype: int64

Total classified sessions:
66422

Negative counts:
movie_viewing_count             0
tv_show_viewing_count           0
limited_series_viewing_count    0
documentary_viewing_count       0
stand_up_viewing_count          0
dtype: int64


In [71]:
# Step 39.6 – Merge content features into the main customer dataset.
session_check = customer_content_features[
    ["customer_id", "total_content_sessions"]
].merge(
    customer_features[["customer_id", "total_viewing_sessions"]],
    on="customer_id",
    how="inner",
    validate="one_to_one"
)

print(
    "Matching session counts:",
    (
        session_check["total_content_sessions"]
        == session_check["total_viewing_sessions"]
    ).sum()
)

print(
    "Mismatched session counts:",
    (
        session_check["total_content_sessions"]
        != session_check["total_viewing_sessions"]
    ).sum()
)

content_features_to_merge = customer_content_features.drop(
    columns=["total_content_sessions"]
)

customer_features = customer_features.merge(
    content_features_to_merge,
    on="customer_id",
    how="left",
    validate="one_to_one"
)

content_count_columns = [
    "unique_genres_watched",
    "movie_viewing_count",
    "tv_show_viewing_count",
    "limited_series_viewing_count",
    "documentary_viewing_count",
    "stand_up_viewing_count",
    "unique_content_languages_watched",
    "unique_maturity_ratings_watched",
    "most_watched_genre_count",
    "preferred_genre_share"
]

customer_features[content_count_columns] = (
    customer_features[content_count_columns].fillna(0)
)

customer_features["has_content_history"] = (
    customer_features["total_viewing_sessions"] > 0
).astype(int)

print("\nUpdated customer features shape:", customer_features.shape)
print("Unique customers:", customer_features["customer_id"].nunique())
print("Duplicate customer IDs:", customer_features["customer_id"].duplicated().sum())

print("\nContent history distribution:")
print(customer_features["has_content_history"].value_counts())

print("\nMissing values in content features:")
print(customer_features[content_features_to_merge.columns].isna().sum())

Matching session counts: 7980
Mismatched session counts: 0

Updated customer features shape: (8000, 78)
Unique customers: 8000
Duplicate customer IDs: 0

Content history distribution:
has_content_history
1    7980
0      20
Name: count, dtype: int64

Missing values in content features:
customer_id                          0
unique_genres_watched                0
movie_viewing_count                  0
tv_show_viewing_count                0
preferred_content_type              20
preferred_content_language          20
unique_content_languages_watched     0
unique_maturity_ratings_watched      0
most_watched_genre                  20
most_watched_genre_count             0
preferred_genre_share                0
limited_series_viewing_count         0
documentary_viewing_count            0
stand_up_viewing_count               0
dtype: int64


In [72]:
# Step 40 – Subscription plan enrichment
# Step 40.1 – Inspect subscription plan structure
print("Subscription plans shape:", cleaned_datasets["subscription_plans"].shape)

print("\nSubscription plan columns:")
print(cleaned_datasets["subscription_plans"].columns.tolist())

print("\nSubscription plan records:")
display(cleaned_datasets["subscription_plans"])

print("\nSubscription plan ID duplicates:")
print(cleaned_datasets["subscription_plans"]["plan_id"].duplicated().sum())

print("\nSubscription plan IDs in subscriptions:")
print(subscriptions["plan_id"].value_counts(dropna=False))

Subscription plans shape: (3, 6)

Subscription plan columns:
['plan_id', 'plan_name', 'monthly_price', 'video_quality', 'max_devices', 'advertisements']

Subscription plan records:


,plan_id,plan_name,monthly_price,video_quality,max_devices,advertisements
0,PLAN01,Basic,6.99,720p (HD),1,Yes
1,PLAN02,Standard,15.49,1080p (Full HD),2,No
2,PLAN03,Premium,22.99,4K (Ultra HD),4,No



Subscription plan ID duplicates:
0

Subscription plan IDs in subscriptions:
plan_id
PLAN01    3671
PLAN02    3301
PLAN03    1903
Name: count, dtype: int64


In [73]:
# Step 40.2 – Join subscription records with plan details
subscription_plan_details = subscriptions.merge(
    cleaned_datasets["subscription_plans"][
        [
            "plan_id",
            "plan_name",
            "video_quality",
            "max_devices",
            "advertisements"
        ]
    ],
    on="plan_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

print("Enriched subscriptions shape:", subscription_plan_details.shape)

print("\nPlan matching status:")
print(subscription_plan_details["_merge"].value_counts())

print("\nPlan distribution:")
print(subscription_plan_details["plan_name"].value_counts(dropna=False))

print("\nMissing plan attributes:")
print(
    subscription_plan_details[
        ["plan_name", "video_quality", "max_devices", "advertisements"]
    ].isna().sum()
)

display(subscription_plan_details.head())

Enriched subscriptions shape: (8875, 15)

Plan matching status:
_merge
both          8875
left_only        0
right_only       0
Name: count, dtype: int64

Plan distribution:
plan_name
Basic       3671
Standard    3301
Premium     1903
Name: count, dtype: int64

Missing plan attributes:
plan_name         0
video_quality     0
max_devices       0
advertisements    0
dtype: int64


,subscription_id,customer_id,plan_id,subscription_start_date,subscription_end_date,subscription_status,auto_renew,cancellation_date,cancellation_reason,monthly_price,plan_name,video_quality,max_devices,advertisements,_merge
0,SUB0000001,CUST000001,PLAN03,2022-04-30,NaT,Active,True,NaT,NaN,22.99,Premium,4K (Ultra HD),4,No,both
1,SUB0000002,CUST000002,PLAN02,2025-12-24,NaT,Active,True,NaT,NaN,15.49,Standard,1080p (Full HD),2,No,both
2,SUB0000003,CUST000003,PLAN03,2019-12-15,2026-04-20,Cancelled,True,2026-04-20,Too expensive,22.99,Premium,4K (Ultra HD),4,No,both
3,SUB0000004,CUST000004,PLAN01,2019-01-28,NaT,Active,False,NaT,NaN,6.99,Basic,720p (HD),1,Yes,both
4,SUB0000005,CUST000005,PLAN01,2024-07-24,NaT,Active,True,NaT,NaN,6.99,Basic,720p (HD),1,Yes,both


In [74]:
# Step 40.3 – Identify each customer's latest subscription plan
subscription_plan_details = subscription_plan_details.drop(
    columns=["_merge"]
)

latest_customer_plan = (
    subscription_plan_details
    .sort_values(
        ["customer_id", "subscription_start_date", "subscription_id"]
    )
    .drop_duplicates(subset="customer_id", keep="last")
    .copy()
)

print("Latest customer plan shape:", latest_customer_plan.shape)

print(
    "Unique customers:",
    latest_customer_plan["customer_id"].nunique()
)

print(
    "Duplicate customer IDs:",
    latest_customer_plan["customer_id"].duplicated().sum()
)

print("\nLatest subscription status distribution:")
print(latest_customer_plan["subscription_status"].value_counts())

print("\nLatest plan distribution:")
print(latest_customer_plan["plan_name"].value_counts())

display(
    latest_customer_plan[
        [
            "customer_id",
            "subscription_id",
            "subscription_start_date",
            "subscription_status",
            "plan_name",
            "monthly_price",
            "video_quality",
            "max_devices",
            "advertisements"
        ]
    ].head(10)
)

Latest customer plan shape: (8000, 14)
Unique customers: 8000
Duplicate customer IDs: 0

Latest subscription status distribution:
subscription_status
Active       6009
Cancelled    1991
Name: count, dtype: int64

Latest plan distribution:
plan_name
Basic       3368
Standard    3018
Premium     1614
Name: count, dtype: int64


,customer_id,subscription_id,subscription_start_date,subscription_status,plan_name,monthly_price,video_quality,max_devices,advertisements
0,CUST000001,SUB0000001,2022-04-30,Active,Premium,22.99,4K (Ultra HD),4,No
1,CUST000002,SUB0000002,2025-12-24,Active,Standard,15.49,1080p (Full HD),2,No
2,CUST000003,SUB0000003,2019-12-15,Cancelled,Premium,22.99,4K (Ultra HD),4,No
3,CUST000004,SUB0000004,2019-01-28,Active,Basic,6.99,720p (HD),1,Yes
4,CUST000005,SUB0000005,2024-07-24,Active,Basic,6.99,720p (HD),1,Yes
5,CUST000006,SUB0000006,2025-07-19,Active,Premium,22.99,4K (Ultra HD),4,No
6,CUST000007,SUB0000007,2025-10-30,Cancelled,Standard,15.49,1080p (Full HD),2,No
7,CUST000008,SUB0000008,2023-08-21,Active,Standard,15.49,1080p (Full HD),2,No
8,CUST000009,SUB0000009,2019-11-18,Active,Standard,15.49,1080p (Full HD),2,No
9,CUST000010,SUB0000010,2021-12-03,Active,Basic,6.99,720p (HD),1,Yes


In [75]:
# Step 40.4 – Create customer-level plan features
latest_plan_features = latest_customer_plan[
    [
        "customer_id",
        "plan_id",
        "plan_name",
        "monthly_price",
        "video_quality",
        "max_devices",
        "advertisements"
    ]
].copy()

latest_plan_features = latest_plan_features.rename(
    columns={
        "plan_id": "latest_plan_id",
        "plan_name": "latest_plan_name",
        "monthly_price": "latest_plan_monthly_price",
        "video_quality": "latest_video_quality",
        "max_devices": "latest_max_devices",
        "advertisements": "latest_plan_advertisements"
    }
)

print("Latest plan features shape:", latest_plan_features.shape)

print("\nMissing values:")
print(latest_plan_features.isna().sum())

print("\nPlan distribution:")
print(latest_plan_features["latest_plan_name"].value_counts())

Latest plan features shape: (8000, 7)

Missing values:
customer_id                   0
latest_plan_id                0
latest_plan_name              0
latest_plan_monthly_price     0
latest_video_quality          0
latest_max_devices            0
latest_plan_advertisements    0
dtype: int64

Plan distribution:
latest_plan_name
Basic       3368
Standard    3018
Premium     1614
Name: count, dtype: int64


In [76]:
# Step 40.5 – Merge plan features into customer features
customer_features = customer_features.merge(
    latest_plan_features,
    on="customer_id",
    how="left",
    validate="one_to_one"
)

print("Updated customer features shape:", customer_features.shape)
print("Unique customers:", customer_features["customer_id"].nunique())
print("Duplicate customer IDs:", customer_features["customer_id"].duplicated().sum())

plan_columns = [
    "latest_plan_id",
    "latest_plan_name",
    "latest_plan_monthly_price",
    "latest_video_quality",
    "latest_max_devices",
    "latest_plan_advertisements"
]

print("\nMissing plan values:")
print(customer_features[plan_columns].isna().sum())

print("\nLatest plan distribution:")
print(customer_features["latest_plan_name"].value_counts())

Updated customer features shape: (8000, 84)


Unique customers: 8000
Duplicate customer IDs: 0

Missing plan values:
latest_plan_id                0
latest_plan_name              0
latest_plan_monthly_price     0
latest_video_quality          0
latest_max_devices            0
latest_plan_advertisements    0
dtype: int64

Latest plan distribution:
latest_plan_name
Basic       3368
Standard    3018
Premium     1614
Name: count, dtype: int64


In [77]:
# Step 41 – Monthly Watch-Time Trends
monthly_viewing_summary = (
    viewing_activity.assign(
        viewing_month=viewing_activity["viewing_date"].dt.to_period("M")
    )
    .groupby("viewing_month")
    .agg(
        total_viewing_sessions=("viewing_id", "nunique"),
        total_watch_minutes=("watch_duration_minutes", "sum"),
        active_customers=("customer_id", "nunique")
    )
    .reset_index()
)

display(monthly_viewing_summary)

,viewing_month,total_viewing_sessions,total_watch_minutes,active_customers
0,2025-07,4829,236666,3359
1,2025-08,4778,237115,3330
2,2025-09,4697,232901,3291
3,2025-10,5144,249998,3542
4,2025-11,4994,246332,3495
5,2025-12,5141,251427,3523
6,2026-01,5321,262021,3672
7,2026-02,5061,252908,3513
8,2026-03,4384,214581,3210
9,2026-04,4419,216495,3191


In [78]:
# Step 41.2 – Analyse the last six months
recent_monthly_summary = monthly_viewing_summary[
    monthly_viewing_summary["viewing_month"].between(
        pd.Period("2026-01", freq="M"),
        pd.Period("2026-06", freq="M")
    )
].copy()

recent_monthly_summary["days_in_month"] = (
    recent_monthly_summary["viewing_month"].dt.days_in_month
)

recent_monthly_summary["average_watch_minutes_per_session"] = (
    recent_monthly_summary["total_watch_minutes"]
    / recent_monthly_summary["total_viewing_sessions"]
)

recent_monthly_summary["average_sessions_per_day"] = (
    recent_monthly_summary["total_viewing_sessions"]
    / recent_monthly_summary["days_in_month"]
)

display(recent_monthly_summary)

,viewing_month,total_viewing_sessions,total_watch_minutes,active_customers,days_in_month,average_watch_minutes_per_session,average_sessions_per_day
6,2026-01,5321,262021,3672,31,49.242812,171.645161
7,2026-02,5061,252908,3513,28,49.971942,180.750000
8,2026-03,4384,214581,3210,31,48.946396,141.419355
9,2026-04,4419,216495,3191,30,48.991853,147.300000
10,2026-05,8804,441974,4302,31,50.201499,284.000000
11,2026-06,8850,441078,4235,30,49.839322,295.000000


In [79]:
# Step 41.3 – Calculate monthly watch time for each customer
recent_months = pd.period_range(
    start="2026-01",
    end="2026-06",
    freq="M"
)

recent_viewing_activity = viewing_activity[
    viewing_activity["viewing_date"].between(
        "2026-01-01",
        "2026-06-30"
    )
].copy()

recent_viewing_activity["viewing_month"] = (
    recent_viewing_activity["viewing_date"].dt.to_period("M")
)

customer_monthly_watch_time = (
    recent_viewing_activity
    .groupby(["customer_id", "viewing_month"])["watch_duration_minutes"]
    .sum()
    .unstack(fill_value=0)
    .reindex(columns=recent_months, fill_value=0)
    .reindex(customer_features["customer_id"], fill_value=0)
)

print("Monthly watch-time matrix shape:", customer_monthly_watch_time.shape)

print(
    "Total watch minutes:",
    customer_monthly_watch_time.sum().sum()
)

print(
    "Expected watch minutes:",
    recent_monthly_summary["total_watch_minutes"].sum()
)

print(
    "Missing values:",
    customer_monthly_watch_time.isna().sum().sum()
)

display(customer_monthly_watch_time.head())

Monthly watch-time matrix shape: (8000, 6)
Total watch minutes: 1829057
Expected watch minutes: 1829057
Missing values: 0


,2026-01,2026-02,2026-03,2026-04,2026-05,2026-06
customer_id,,,,,,
CUST000744,91,0,0,0,0,55
CUST001502,98,70,84,57,0,0
CUST005383,91,0,0,0,0,0
CUST004708,37,0,0,0,0,0
CUST005877,102,0,106,78,0,0


In [80]:
# Step 41.4 – Calculate monthly watch-time trend
month_positions = np.arange(len(recent_months))

customer_watch_time_trend = customer_monthly_watch_time.apply(
    lambda row: np.polyfit(
        month_positions,
        row.to_numpy(dtype=float),
        1
    )[0],
    axis=1
)

customer_watch_trend_features = pd.DataFrame({
    "customer_id": customer_monthly_watch_time.index,
    "monthly_watch_time_trend": customer_watch_time_trend.values
})

print("Trend features shape:", customer_watch_trend_features.shape)

print("\nTrend statistics:")
display(customer_watch_trend_features["monthly_watch_time_trend"].describe())

print("\nTrend categories:")
print(
    pd.Series(
        np.select(
            [
                customer_watch_trend_features["monthly_watch_time_trend"] > 0,
                customer_watch_trend_features["monthly_watch_time_trend"] < 0
            ],
            ["Increasing", "Decreasing"],
            default="Stable"
        )
    ).value_counts()
)

print("\nMissing trend values:", customer_watch_trend_features["monthly_watch_time_trend"].isna().sum())

display(customer_watch_trend_features.head())

Trend features shape: (8000, 2)

Trend statistics:


count    8000.000000
mean        5.229989
std        20.654444
min      -132.285714
25%        -6.257143
50%         0.800000
75%        13.000000
max       257.857143
Name: monthly_watch_time_trend, dtype: float64


Trend categories:
Increasing    4177
Decreasing    3511
Stable         312
Name: count, dtype: int64

Missing trend values: 0


,customer_id,monthly_watch_time_trend
0,CUST000744,-5.142857
1,CUST001502,-20.771429
2,CUST005383,-13.000000
3,CUST004708,-5.285714
4,CUST005877,-15.371429


In [81]:
# Step 41.5 – Merge watch-time trends
customer_features = customer_features.merge(
    customer_watch_trend_features,
    on="customer_id",
    how="left",
    validate="one_to_one"
)

print("Updated customer features shape:", customer_features.shape)
print("Unique customers:", customer_features["customer_id"].nunique())
print("Duplicate customer IDs:", customer_features["customer_id"].duplicated().sum())

print("\nMissing trend values:")
print(customer_features["monthly_watch_time_trend"].isna().sum())

print("\nTrend statistics:")
display(customer_features["monthly_watch_time_trend"].describe())

Updated customer features shape: (8000, 85)
Unique customers: 8000
Duplicate customer IDs: 0

Missing trend values:
0

Trend statistics:


count    8000.000000
mean        5.229989
std        20.654444
min      -132.285714
25%        -6.257143
50%         0.800000
75%        13.000000
max       257.857143
Name: monthly_watch_time_trend, dtype: float64

In [82]:
# Step 42 – Login Frequency and Engagement Features
# Step 42.1 – Calculate active months
viewing_activity["viewing_month"] = (viewing_activity["viewing_date"].dt.to_period("M"))

customer_activity_frequency = (
    viewing_activity.groupby("customer_id")
    .agg(
        active_months=("viewing_month", "nunique"),
        total_sessions=("viewing_id", "nunique")
    )
    .reset_index()
)

customer_activity_frequency = (
    customer_activity_frequency.set_index("customer_id")
    .reindex(customer_features["customer_id"])
    .fillna(0)
    .reset_index()
)

customer_activity_frequency["sessions_per_active_month"] = (
    customer_activity_frequency["total_sessions"]
    / customer_activity_frequency["active_months"].replace(0, np.nan)
).fillna(0)

customer_activity_frequency["active_month_rate"] = (
    customer_activity_frequency["active_months"] / 12
)

print("Activity frequency shape:", customer_activity_frequency.shape)

print("\nActive months distribution:")
print(customer_activity_frequency["active_months"].value_counts().sort_index())

print("\nActivity frequency statistics:")
display(
    customer_activity_frequency[
        ["active_months", "sessions_per_active_month", "active_month_rate"]
    ].describe()
)

print("\nTotal sessions:", customer_activity_frequency["total_sessions"].sum())
print("Missing values:", customer_activity_frequency.isna().sum().sum())

display(customer_activity_frequency.head())

Activity frequency shape: (8000, 5)

Active months distribution:
active_months
0.0       20
1.0      312
2.0      616
3.0      655
4.0     1040
5.0     1399
6.0     1529
7.0     1222
8.0      744
9.0      333
10.0     109
11.0      20
12.0       1
Name: count, dtype: int64

Activity frequency statistics:


,active_months,sessions_per_active_month,active_month_rate
count,8000.000000,8000.000000,8000.000000
mean,5.332875,1.598417,0.444406
std,2.113913,0.663806,0.176159
min,0.000000,0.000000,0.000000
25%,4.000000,1.200000,0.333333
50%,5.000000,1.500000,0.416667
75%,7.000000,1.756944,0.583333
max,12.000000,15.000000,1.000000



Total sessions: 66422.0
Missing values: 0


,customer_id,active_months,total_sessions,sessions_per_active_month,active_month_rate
0,CUST000744,4.0,5.0,1.250000,0.333333
1,CUST001502,4.0,5.0,1.250000,0.333333
2,CUST005383,3.0,6.0,2.000000,0.250000
3,CUST004708,7.0,11.0,1.571429,0.583333
4,CUST005877,3.0,5.0,1.666667,0.250000


In [83]:
# Step 42.2 – Merge activity frequency features
customer_activity_frequency["active_months"] = (
    customer_activity_frequency["active_months"].astype(int)
)

customer_activity_frequency["total_sessions"] = (
    customer_activity_frequency["total_sessions"].astype(int)
)

customer_features = customer_features.merge(
    customer_activity_frequency[
        [
            "customer_id",
            "active_months",
            "sessions_per_active_month",
            "active_month_rate"
        ]
    ],
    on="customer_id",
    how="left",
    validate="one_to_one"
)

session_counts_match = customer_features["customer_id"].map(
    customer_activity_frequency.set_index("customer_id")["total_sessions"]
).eq(customer_features["total_viewing_sessions"]).all()

print("Customer features shape:", customer_features.shape)
print("Unique customers:", customer_features["customer_id"].nunique())
print("Duplicate customer IDs:", customer_features["customer_id"].duplicated().sum())
print("\nMissing values in new features:")
print(
    customer_features[
        ["active_months", "sessions_per_active_month", "active_month_rate"]
    ].isna().sum()
)
print("\nSession counts match:", session_counts_match)

Customer features shape: (8000, 88)
Unique customers: 8000
Duplicate customer IDs: 0

Missing values in new features:
active_months                0
sessions_per_active_month    0
active_month_rate            0
dtype: int64

Session counts match: True


In [84]:
# Step 43.1 – Calculate payment failure rate
customer_features["payment_failure_rate"] = (
    customer_features["failed_payment_count"]
    / customer_features["total_payment_transactions"].replace(0, np.nan)
).fillna(0)

print("Payment failure rate statistics:")
display(customer_features["payment_failure_rate"].describe())

print("\nPayment failure rate range:")
print("Minimum:", customer_features["payment_failure_rate"].min())
print("Maximum:", customer_features["payment_failure_rate"].max())

print("\nMissing values:", customer_features["payment_failure_rate"].isna().sum())

display(
    customer_features[
        ["customer_id", "total_payment_transactions",
         "failed_payment_count", "payment_failure_rate"]
    ].head()
)

Payment failure rate statistics:


count    8000.000000
mean        0.065712
std         0.082466
min         0.000000
25%         0.000000
50%         0.066667
75%         0.111111
max         1.000000
Name: payment_failure_rate, dtype: float64


Payment failure rate range:
Minimum: 0.0
Maximum: 1.0

Missing values: 0


,customer_id,total_payment_transactions,failed_payment_count,payment_failure_rate
0,CUST000744,15,2,0.133333
1,CUST001502,5,0,0.000000
2,CUST005383,15,1,0.066667
3,CUST004708,15,1,0.066667
4,CUST005877,6,1,0.166667


In [85]:
# Step 44.1 – Create a customer engagement score
customer_features["engagement_recency_score"] = (
    1 - customer_features["days_since_last_viewing"].fillna(365) / 365
).clip(0, 1)

customer_features["engagement_frequency_score"] = (
    customer_features["active_month_rate"]
).clip(0, 1)

customer_features["engagement_watch_depth_score"] = (
    customer_features["average_completion_percentage"].fillna(0) / 100
).clip(0, 1)

customer_features["customer_engagement_score"] = (
    (
        customer_features["engagement_recency_score"]
        + customer_features["engagement_frequency_score"]
        + customer_features["engagement_watch_depth_score"]
    ) / 3
) * 100

print("Engagement score statistics:")
display(
    customer_features[
        [
            "engagement_recency_score",
            "engagement_frequency_score",
            "engagement_watch_depth_score",
            "customer_engagement_score"
        ]
    ].describe()
)

print("\nEngagement score range:")
print("Minimum:", customer_features["customer_engagement_score"].min())
print("Maximum:", customer_features["customer_engagement_score"].max())

print("\nMissing values:")
print(
    customer_features[
        [
            "engagement_recency_score",
            "engagement_frequency_score",
            "engagement_watch_depth_score",
            "customer_engagement_score"
        ]
    ].isna().sum()
)

display(
    customer_features[
        [
            "customer_id",
            "engagement_recency_score",
            "engagement_frequency_score",
            "engagement_watch_depth_score",
            "customer_engagement_score"
        ]
    ].head()
)

Engagement score statistics:


,engagement_recency_score,engagement_frequency_score,engagement_watch_depth_score,customer_engagement_score
count,8000.000000,8000.000000,8000.000000,8000.000000
mean,0.865477,0.444406,0.702241,67.070799
std,0.163434,0.176159,0.091509,10.075124
min,0.000000,0.000000,0.000000,0.000000
25%,0.813699,0.333333,0.655000,62.937215
50%,0.928767,0.416667,0.705670,68.635274
75%,0.978082,0.583333,0.754750,73.560964
max,1.000000,1.000000,1.000000,90.000358



Engagement score range:
Minimum: 0.0
Maximum: 90.00035813412123

Missing values:
engagement_recency_score        0
engagement_frequency_score      0
engagement_watch_depth_score    0
customer_engagement_score       0
dtype: int64


,customer_id,engagement_recency_score,engagement_frequency_score,engagement_watch_depth_score,customer_engagement_score
0,CUST000744,0.969863,0.333333,0.702,66.839878
1,CUST001502,0.832877,0.333333,0.728,63.140335
2,CUST005383,0.583562,0.250000,0.765,53.285388
3,CUST004708,0.558904,0.583333,0.640,59.407915
4,CUST005877,0.791781,0.250000,0.608,54.992694


In [86]:
# Step 44.2 – Validate customers without viewing activity
no_viewing_customers = customer_features[customer_features["has_viewing_activity"] == 0]

print("Customers without viewing activity:", len(no_viewing_customers))

print("\nEngagement scores for customers without viewing activity:")
display(
    no_viewing_customers[
        [
            "customer_id",
            "engagement_recency_score",
            "engagement_frequency_score",
            "engagement_watch_depth_score",
            "customer_engagement_score"
        ]
    ].head()
)

print(
    "\nAll no-viewing customers have zero engagement score:",
    (no_viewing_customers["customer_engagement_score"] == 0).all()
)

print(
    "\nAll engagement scores within 0-100:",
    customer_features["customer_engagement_score"].between(0, 100).all()
)

Customers without viewing activity: 20

Engagement scores for customers without viewing activity:


,customer_id,engagement_recency_score,engagement_frequency_score,engagement_watch_depth_score,customer_engagement_score
280,CUST005845,0.0,0.0,0.0,0.0
388,CUST005866,0.0,0.0,0.0,0.0
755,CUST006781,0.0,0.0,0.0,0.0
1626,CUST003140,0.0,0.0,0.0,0.0
2590,CUST004314,0.0,0.0,0.0,0.0



All no-viewing customers have zero engagement score: True

All engagement scores within 0-100: True


In [87]:
# Step 45.1 – Final feature engineering validation
import numpy as np

numeric_features = customer_features.select_dtypes(include=np.number)

print("Final customer features shape:", customer_features.shape)
print("Unique customers:", customer_features["customer_id"].nunique())
print("Duplicate customer IDs:", customer_features["customer_id"].duplicated().sum())
print("Duplicate column names:", customer_features.columns.duplicated().sum())

print("\nMissing values by column:")
missing_summary = customer_features.isna().sum()
display(missing_summary[missing_summary > 0].sort_values(ascending=False))

print("\nTotal missing values:", customer_features.isna().sum().sum())

print(
    "\nInfinite numeric values:",
    np.isinf(numeric_features).sum().sum()
)

print("\nNew feature ranges:")
display(
    customer_features[
        [
            "payment_failure_rate",
            "active_months",
            "sessions_per_active_month",
            "active_month_rate",
            "customer_engagement_score"
        ]
    ].describe()
)

Final customer features shape: (8000, 93)
Unique customers: 8000
Duplicate customer IDs: 0
Duplicate column names: 0

Missing values by column:


state_or_region                       8000
avg_customer_satisfaction             4460
average_resolution_time_hours         4460
last_feedback_date                    4439
average_feedback_rating               4439
days_since_last_feedback              4439
preferred_content_type                  20
days_since_last_viewing                 20
last_viewing_date                       20
preferred_content_language              20
most_watched_genre                      20
days_since_last_successful_payment      10
last_successful_payment_date            10
dtype: int64


Total missing values: 30357

Infinite numeric values: 0

New feature ranges:


,payment_failure_rate,active_months,sessions_per_active_month,active_month_rate,customer_engagement_score
count,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000
mean,0.065712,5.332875,1.598417,0.444406,67.070799
std,0.082466,2.113913,0.663806,0.176159,10.075124
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,4.000000,1.200000,0.333333,62.937215
50%,0.066667,5.000000,1.500000,0.416667,68.635274
75%,0.111111,7.000000,1.756944,0.583333,73.560964
max,1.000000,12.000000,15.000000,1.000000,90.000358


In [88]:
# Step 46.1 – Inspect churn labels
churn_labels = cleaned_datasets["churn_labels"].copy()

print("Churn labels shape:", churn_labels.shape)

print("\nColumn names:")
print(churn_labels.columns.tolist())

print("\nData types:")
print(churn_labels.dtypes)

print("\nMissing values:")
print(churn_labels.isna().sum())

print("\nUnique customer IDs:", churn_labels["customer_id"].nunique())
print("Duplicate customer IDs:", churn_labels["customer_id"].duplicated().sum())

print("\nFirst five records:")
display(churn_labels.head())

print("\nDataset information:")
churn_labels.info()

Churn labels shape: (8000, 4)

Column names:
['customer_id', 'churned', 'churn_date', 'churn_reason']

Data types:
customer_id                str
churned                   bool
churn_date      datetime64[us]
churn_reason               str
dtype: object

Missing values:
customer_id        0
churned            0
churn_date      6009
churn_reason    6009
dtype: int64

Unique customer IDs: 8000
Duplicate customer IDs: 0

First five records:


,customer_id,churned,churn_date,churn_reason
0,CUST000001,False,NaT,NaN
1,CUST000002,False,NaT,NaN
2,CUST000003,True,2026-04-20,Too expensive
3,CUST000004,False,NaT,NaN
4,CUST000005,False,NaT,NaN



Dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   customer_id   8000 non-null   str           
 1   churned       8000 non-null   bool          
 2   churn_date    1991 non-null   datetime64[us]
 3   churn_reason  1991 non-null   str           
dtypes: bool(1), datetime64[us](1), str(2)
memory usage: 312.2 KB


In [89]:
# Step 46.2 – Validate churn distribution and dates
print("Churn distribution:")
display(churn_labels["churned"].value_counts())

print("\nChurn percentage:")
display(
    (churn_labels["churned"].value_counts(normalize=True) * 100).round(2)
)

print("\nChurn date range:")
print("Earliest:", churn_labels["churn_date"].min())
print("Latest:", churn_labels["churn_date"].max())

print("\nChurn label and date consistency:")
print(
    "Churned customers with dates:",
    churn_labels.loc[churn_labels["churned"], "churn_date"].notna().sum()
)

print(
    "Non-churned customers without dates:",
    churn_labels.loc[~churn_labels["churned"], "churn_date"].isna().sum()
)

print("\nCustomer ID matching:")
print(
    "All churn-label customers exist in customer_features:",
    churn_labels["customer_id"].isin(customer_features["customer_id"]).all()
)

Churn distribution:


churned
False    6009
True     1991
Name: count, dtype: int64


Churn percentage:


churned
False    75.11
True     24.89
Name: proportion, dtype: float64


Churn date range:
Earliest: 2025-10-03 00:00:00
Latest: 2026-06-30 00:00:00

Churn label and date consistency:
Churned customers with dates: 1991
Non-churned customers without dates: 6009

Customer ID matching:
All churn-label customers exist in customer_features: True


In [90]:
# Step 46.3 – Check for temporal data leakage
churned_customers = churn_labels.loc[
    churn_labels["churned"],
    ["customer_id", "churn_date"]
]

event_tables = {
    "Viewing activity": (cleaned_datasets["viewing_activity"], "viewing_date"),
    "Payments": (cleaned_datasets["payments"], "payment_date"),
    "Support tickets": (cleaned_datasets["support_tickets"], "ticket_date"),
    "Customer feedback": (cleaned_datasets["customer_feedback"], "feedback_date"),
    "Subscriptions": (cleaned_datasets["subscriptions"], "subscription_start_date")
}

for table_name, (df, date_column) in event_tables.items():
    event_check = df[["customer_id", date_column]].merge(
        churned_customers,
        on="customer_id",
        how="inner"
    )

    events_after_churn = event_check[
        event_check[date_column].notna()
        & (event_check[date_column] > event_check["churn_date"])
    ]

    print(f"\n{table_name}")
    print("Events after churn:", len(events_after_churn))
    print("Customers with events after churn:",
          events_after_churn["customer_id"].nunique())


Viewing activity
Events after churn: 5563
Customers with events after churn: 1545

Payments
Events after churn: 1105
Customers with events after churn: 348

Support tickets
Events after churn: 292
Customers with events after churn: 242

Customer feedback
Events after churn: 219
Customers with events after churn: 190

Subscriptions
Events after churn: 42
Customers with events after churn: 42


In [91]:
# Step 46.4 – Set and validate the prediction cutoff
prediction_cutoff_date = pd.Timestamp("2025-09-30")

churned_records = churn_labels[churn_labels["churned"]]

print("Prediction cutoff:", prediction_cutoff_date.date())

print(
    "Earliest churn date:",
    churned_records["churn_date"].min().date()
)

print(
    "Latest churn date:",
    churned_records["churn_date"].max().date()
)

print(
    "All churn events occur after prediction cutoff:",
    (churned_records["churn_date"] > prediction_cutoff_date).all()
)

print(
    "Churned customers:",
    churn_labels["churned"].sum()
)

print(
    "Non-churned customers:",
    (~churn_labels["churned"]).sum()
)

Prediction cutoff: 2025-09-30
Earliest churn date: 2025-10-03
Latest churn date: 2026-06-30
All churn events occur after prediction cutoff: True
Churned customers: 1991
Non-churned customers: 6009


In [92]:
print("cleaned_datasets:", "cleaned_datasets" in globals())
print("customers:", "customers" in globals())
print("churn_labels:", "churn_labels" in globals())

cleaned_datasets: True
customers: False
churn_labels: True


In [93]:
customers = cleaned_datasets["customers"]
churn_labels = cleaned_datasets["churn_labels"]

print("Customers shape:", customers.shape)
print("Churn labels shape:", churn_labels.shape)

Customers shape: (8000, 12)
Churn labels shape: (8000, 4)


In [94]:
from pathlib import Path

print("Current working directory:")
print(Path.cwd())

print("\nCSV files found:")

csv_files = list(Path.cwd().rglob("*.csv"))

for file in csv_files:
    print(file)

print("\nTotal CSV files found:", len(csv_files))

Current working directory:
c:\Users\USER\Documents\Internmo Project1\fae5dd037f3e68b4f411b2243b81c23d\notebooks

CSV files found:

Total CSV files found: 0


In [95]:
from pathlib import Path

project_root = Path.cwd().parent

print("Project root:", project_root)

print("\nCSV files found in project:")

csv_files = list(project_root.rglob("*.csv"))

for file in csv_files:
    print(file)

print("\nTotal CSV files:", len(csv_files))

Project root: c:\Users\USER\Documents\Internmo Project1\fae5dd037f3e68b4f411b2243b81c23d

CSV files found in project:
c:\Users\USER\Documents\Internmo Project1\fae5dd037f3e68b4f411b2243b81c23d\churn_labels.csv
c:\Users\USER\Documents\Internmo Project1\fae5dd037f3e68b4f411b2243b81c23d\content.csv
c:\Users\USER\Documents\Internmo Project1\fae5dd037f3e68b4f411b2243b81c23d\customers (3).csv
c:\Users\USER\Documents\Internmo Project1\fae5dd037f3e68b4f411b2243b81c23d\customer_feedback (2).csv
c:\Users\USER\Documents\Internmo Project1\fae5dd037f3e68b4f411b2243b81c23d\payments (2).csv
c:\Users\USER\Documents\Internmo Project1\fae5dd037f3e68b4f411b2243b81c23d\subscriptions (1).csv
c:\Users\USER\Documents\Internmo Project1\fae5dd037f3e68b4f411b2243b81c23d\subscription_plans.csv
c:\Users\USER\Documents\Internmo Project1\fae5dd037f3e68b4f411b2243b81c23d\support_tickets (1).csv
c:\Users\USER\Documents\Internmo Project1\fae5dd037f3e68b4f411b2243b81c23d\viewing_activity.csv
c:\Users\USER\Documents\Int

In [96]:
from pathlib import Path
import pandas as pd

project_root = Path.cwd().parent
final_data_dir = project_root / "data" / "processed" / "final"

dataset_files = {
    "customers": "customers_final.csv",
    "subscription_plans": "subscription_plans_final.csv",
    "subscriptions": "subscriptions_final1.csv",
    "content": "content_final.csv",
    "viewing_activity": "viewing_activity_final.csv",
    "payments": "payments_final1.csv",
    "support_tickets": "support_tickets_final1.csv",
    "customer_feedback": "customer_feedback_final1.csv",
    "churn_labels": "churn_labels_final.csv"
}

missing_files = [
    filename
    for filename in dataset_files.values()
    if not (final_data_dir / filename).exists()
]

if missing_files:
    raise FileNotFoundError(
        f"Missing final datasets: {missing_files}"
    )

cleaned_datasets = {
    name: pd.read_csv(final_data_dir / filename)
    for name, filename in dataset_files.items()
}

customers = cleaned_datasets["customers"]
subscription_plans = cleaned_datasets["subscription_plans"]
subscriptions = cleaned_datasets["subscriptions"]
content = cleaned_datasets["content"]
viewing_activity = cleaned_datasets["viewing_activity"]
payments = cleaned_datasets["payments"]
support_tickets = cleaned_datasets["support_tickets"]
customer_feedback = cleaned_datasets["customer_feedback"]
churn_labels = cleaned_datasets["churn_labels"]

print("All final datasets loaded successfully!")
print("\nDataset summary:")

for name, df in cleaned_datasets.items():
    print(f"{name}: {df.shape}")

All final datasets loaded successfully!

Dataset summary:
customers: (8000, 12)
subscription_plans: (3, 6)
subscriptions: (8875, 10)
content: (500, 9)
viewing_activity: (66422, 9)
payments: (110735, 8)
support_tickets: (6120, 11)
customer_feedback: (5046, 6)
churn_labels: (8000, 4)


In [97]:
# Step 46.5 – Check the Revised Cohort
prediction_cutoff_date = pd.Timestamp("2026-05-31")
label_window_end = pd.Timestamp("2026-06-30")

customers["registration_date"] = pd.to_datetime(
    customers["registration_date"],
    errors="coerce"
)

churn_labels["churn_date"] = pd.to_datetime(
    churn_labels["churn_date"],
    errors="coerce"
)

churn_cohort = customers.loc[
    customers["registration_date"] <= prediction_cutoff_date,
    ["customer_id"]
].merge(
    churn_labels[["customer_id", "churn_date"]],
    on="customer_id",
    how="left",
    validate="one_to_one"
)

eligible_cohort = churn_cohort[
    churn_cohort["churn_date"].isna()
    | (churn_cohort["churn_date"] > prediction_cutoff_date)
].copy()

eligible_cohort["churned_next_month"] = (
    (eligible_cohort["churn_date"] > prediction_cutoff_date)
    & (eligible_cohort["churn_date"] <= label_window_end)
)

print("Registered customers:", len(churn_cohort))
print("Eligible customers:", len(eligible_cohort))

print("\nNext-month churn distribution:")
print(eligible_cohort["churned_next_month"].value_counts())

print("\nChurn percentage:")
print(
    (eligible_cohort["churned_next_month"].value_counts(normalize=True) * 100).round(2)
)

Registered customers: 8000
Eligible customers: 6260

Next-month churn distribution:
churned_next_month
False    6009
True      251
Name: count, dtype: int64

Churn percentage:
churned_next_month
False    95.99
True      4.01
Name: proportion, dtype: float64


In [98]:
# Restore the feature engineering variables
required_variables = [
    "churn_target",
    "churn_features",
    "feedback_asof",
    "customer_feedback_asof"
]

for variable in required_variables:
    print(f"{variable}: {variable in globals()}")

churn_target: False
churn_features: False
feedback_asof: False
customer_feedback_asof: False


In [99]:
# Step 46.6
churn_target = eligible_cohort[
    ["customer_id", "churned_next_month"]
].copy()

churn_target = churn_target.rename(
    columns={"churned_next_month": "churned"}
)

print("Churn target shape:", churn_target.shape)
print("\nTarget distribution:")
print(churn_target["churned"].value_counts())
print("\nMissing values:", churn_target.isna().sum().sum())
print("Duplicate customer IDs:", churn_target["customer_id"].duplicated().sum())

Churn target shape: (6260, 2)

Target distribution:
churned
False    6009
True      251
Name: count, dtype: int64

Missing values: 0
Duplicate customer IDs: 0


In [100]:
# 47.1 – Create the Customer Base
churn_features = eligible_cohort[["customer_id"]].merge(
    customers,
    on="customer_id",
    how="left",
    validate="one_to_one"
)

churn_features["registration_date"] = pd.to_datetime(
    churn_features["registration_date"],
    errors="coerce"
)

churn_features["tenure_days"] = (
    prediction_cutoff_date - churn_features["registration_date"]
).dt.days

churn_features = churn_features.drop(
    columns=["registration_date", "state_or_region"]
)

print("Churn features shape:", churn_features.shape)
print("Unique customers:", churn_features["customer_id"].nunique())
print("Duplicate customer IDs:", churn_features["customer_id"].duplicated().sum())
print("Missing customer IDs:", churn_features["customer_id"].isna().sum())

print("\nTenure summary:")
print(churn_features["tenure_days"].describe())

print("\nNegative tenure values:", (churn_features["tenure_days"] < 0).sum())

Churn features shape: (6260, 11)
Unique customers: 6260
Duplicate customer IDs: 0
Missing customer IDs: 0

Tenure summary:
count    6260.000000
mean     1365.726358
std       777.850061
min         0.000000
25%       699.750000
50%      1375.000000
75%      2026.000000
max      2707.000000
Name: tenure_days, dtype: float64

Negative tenure values: 0


In [101]:
# Step 47.2 – Filter Viewing Activity as of May 31, 2026
viewing_asof = viewing_activity.copy()

viewing_asof["viewing_date"] = pd.to_datetime(
    viewing_asof["viewing_date"],
    errors="coerce"
)

viewing_asof = viewing_asof[
    viewing_asof["viewing_date"].notna()
    & (viewing_asof["viewing_date"] <= prediction_cutoff_date)
].copy()

print("Viewing records:", len(viewing_asof))
print("Unique customers:", viewing_asof["customer_id"].nunique())
print("Earliest viewing date:", viewing_asof["viewing_date"].min())
print("Latest viewing date:", viewing_asof["viewing_date"].max())
print(
    "Records after cutoff:",
    (viewing_asof["viewing_date"] > prediction_cutoff_date).sum()
)
print("Missing viewing dates:", viewing_asof["viewing_date"].isna().sum())

Viewing records: 57572
Unique customers: 7935
Earliest viewing date: 2025-07-01 00:00:00
Latest viewing date: 2026-05-31 00:00:00
Records after cutoff: 0
Missing viewing dates: 0


In [102]:
# Step 47.3 – Aggregate Customer-Level Viewing Features
viewing_asof = viewing_asof[
    viewing_asof["customer_id"].isin(eligible_cohort["customer_id"])
].copy()

customer_viewing_asof = (
    viewing_asof
    .groupby("customer_id")
    .agg(
        total_viewing_sessions=("viewing_id", "count"),
        total_watch_duration_minutes=("watch_duration_minutes", "sum"),
        average_watch_duration_minutes=("watch_duration_minutes", "mean"),
        average_completion_percentage=("completion_percentage", "mean"),
        average_session_duration_minutes=("session_duration_minutes", "mean"),
        unique_content_watched=("content_id", "nunique"),
        active_viewing_days=("viewing_date", "nunique"),
        last_viewing_date=("viewing_date", "max")
    )
    .reset_index()
)

customer_viewing_asof["days_since_last_viewing"] = (
    prediction_cutoff_date -
    customer_viewing_asof["last_viewing_date"]
).dt.days

print("Customer viewing features shape:", customer_viewing_asof.shape)
print("Unique customers:", customer_viewing_asof["customer_id"].nunique())
print("Total viewing sessions:", customer_viewing_asof["total_viewing_sessions"].sum())
print("Duplicate customer IDs:", customer_viewing_asof["customer_id"].duplicated().sum())

print("\nMissing values:")
print(customer_viewing_asof.isna().sum())

print("\nViewing feature summary:")
print(customer_viewing_asof.describe())

Customer viewing features shape: (6218, 10)
Unique customers: 6218
Total viewing sessions: 45107
Duplicate customer IDs: 0

Missing values:
customer_id                         0
total_viewing_sessions              0
total_watch_duration_minutes        0
average_watch_duration_minutes      0
average_completion_percentage       0
average_session_duration_minutes    0
unique_content_watched              0
active_viewing_days                 0
last_viewing_date                   0
days_since_last_viewing             0
dtype: int64

Viewing feature summary:
       total_viewing_sessions  total_watch_duration_minutes  \
count             6218.000000                    6218.00000   
mean                 7.254262                     357.44146   
min                  1.000000                       3.00000   
25%                  5.000000                     230.00000   
50%                  7.000000                     346.00000   
75%                  9.000000                     458.00000   


In [103]:
# Step 47.4 – Merge Viewing Features
churn_features = churn_features.merge(
    customer_viewing_asof,
    on="customer_id",
    how="left",
    validate="one_to_one"
)

churn_features["has_viewing_activity"] = (
    churn_features["total_viewing_sessions"].notna().astype(int)
)

viewing_count_columns = [
    "total_viewing_sessions",
    "total_watch_duration_minutes",
    "unique_content_watched",
    "active_viewing_days"
]

churn_features[viewing_count_columns] = (
    churn_features[viewing_count_columns].fillna(0)
)

print("Churn features shape:", churn_features.shape)
print("Customers with viewing activity:", churn_features["has_viewing_activity"].sum())
print("Customers without viewing activity:", (churn_features["has_viewing_activity"] == 0).sum())
print("Duplicate customer IDs:", churn_features["customer_id"].duplicated().sum())

print("\nMissing values:")
print(churn_features.isna().sum())

Churn features shape: (6260, 21)
Customers with viewing activity: 6218
Customers without viewing activity: 42
Duplicate customer IDs: 0

Missing values:
customer_id                          0
first_name                           0
last_name                            0
age                                  0
gender                               0
country                              0
city                                 0
acquisition_channel                  0
customer_segment                     0
preferred_language                   0
tenure_days                          0
total_viewing_sessions               0
total_watch_duration_minutes         0
average_watch_duration_minutes      42
average_completion_percentage       42
average_session_duration_minutes    42
unique_content_watched               0
active_viewing_days                  0
last_viewing_date                   42
days_since_last_viewing             42
has_viewing_activity                 0
dtype: int64


In [104]:
# Step 47.5 – Filter Payment Transactions
payments_asof = payments.copy()

payments_asof["payment_date"] = pd.to_datetime(
    payments_asof["payment_date"],
    errors="coerce"
)

payments_asof = payments_asof[
    payments_asof["payment_date"].notna()
    & (payments_asof["payment_date"] <= prediction_cutoff_date)
    & (payments_asof["customer_id"].isin(eligible_cohort["customer_id"]))
].copy()

print("Payment transactions:", len(payments_asof))
print("Unique customers:", payments_asof["customer_id"].nunique())
print("Earliest payment date:", payments_asof["payment_date"].min())
print("Latest payment date:", payments_asof["payment_date"].max())

print("\nPayment status distribution:")
print(payments_asof["payment_status"].value_counts())

print("\nRecords after cutoff:",
      (payments_asof["payment_date"] > prediction_cutoff_date).sum())

Payment transactions: 86664
Unique customers: 6260
Earliest payment date: 2019-01-01 00:00:00
Latest payment date: 2026-05-31 00:00:00

Payment status distribution:
payment_status
Success    81434
Failed      5230
Name: count, dtype: int64

Records after cutoff: 0


In [105]:
# Step 47.6 – Aggregate Payment Features
payment_features_asof = payments_asof.copy()

payment_features_asof["is_successful_payment"] = (
    payment_features_asof["payment_status"] == "Success"
).astype(int)

payment_features_asof["is_failed_payment"] = (
    payment_features_asof["payment_status"] == "Failed"
).astype(int)

payment_features_asof["successful_payment_amount"] = (
    payment_features_asof["amount"].where(
        payment_features_asof["payment_status"] == "Success"
    )
)

payment_features_asof["successful_payment_date"] = (
    payment_features_asof["payment_date"].where(
        payment_features_asof["payment_status"] == "Success"
    )
)

customer_payment_asof = (
    payment_features_asof
    .groupby("customer_id")
    .agg(
        total_payment_transactions=("payment_id", "nunique"),
        successful_payment_count=("is_successful_payment", "sum"),
        failed_payment_count=("is_failed_payment", "sum"),
        total_amount_paid=("successful_payment_amount", "sum"),
        average_successful_payment_amount=("successful_payment_amount", "mean"),
        payment_success_rate=("is_successful_payment", "mean"),
        unique_payment_methods=("payment_method", "nunique"),
        most_used_payment_method=(
            "payment_method",
            lambda x: x.mode().iloc[0] if not x.mode().empty else pd.NA
        ),
        last_payment_date=("payment_date", "max"),
        last_successful_payment_date=("successful_payment_date", "max")
    )
    .reset_index()
)

customer_payment_asof["days_since_last_successful_payment"] = (
    prediction_cutoff_date -
    customer_payment_asof["last_successful_payment_date"]
).dt.days

print("Payment features shape:", customer_payment_asof.shape)
print("Unique customers:", customer_payment_asof["customer_id"].nunique())
print("Total transactions:", customer_payment_asof["total_payment_transactions"].sum())
print("Successful payments:", customer_payment_asof["successful_payment_count"].sum())
print("Failed payments:", customer_payment_asof["failed_payment_count"].sum())
print("Customers with no successful payments:",
      (customer_payment_asof["successful_payment_count"] == 0).sum())
print("Duplicate customer IDs:", customer_payment_asof["customer_id"].duplicated().sum())

print("\nMissing values:")
print(customer_payment_asof.isna().sum())

Payment features shape: (6260, 12)
Unique customers: 6260
Total transactions: 86664
Successful payments: 81434
Failed payments: 5230
Customers with no successful payments: 7
Duplicate customer IDs: 0

Missing values:
customer_id                           0
total_payment_transactions            0
successful_payment_count              0
failed_payment_count                  0
total_amount_paid                     0
average_successful_payment_amount     7
payment_success_rate                  0
unique_payment_methods                0
most_used_payment_method              0
last_payment_date                     0
last_successful_payment_date          7
days_since_last_successful_payment    7
dtype: int64


In [106]:
from pathlib import Path
import pandas as pd

project_root = Path.cwd().parent
final_data_dir = project_root / "data" / "processed" / "final"

dataset_files = {
    "customers": "customers_final.csv",
    "subscription_plans": "subscription_plans_final.csv",
    "subscriptions": "subscriptions_final1.csv",
    "content": "content_final.csv",
    "viewing_activity": "viewing_activity_final.csv",
    "payments": "payments_final1.csv",
    "support_tickets": "support_tickets_final1.csv",
    "customer_feedback": "customer_feedback_final1.csv",
    "churn_labels": "churn_labels_final.csv"
}

cleaned_datasets = {
    name: pd.read_csv(final_data_dir / filename)
    for name, filename in dataset_files.items()
}

customers = cleaned_datasets["customers"]
subscription_plans = cleaned_datasets["subscription_plans"]
subscriptions = cleaned_datasets["subscriptions"]
content = cleaned_datasets["content"]
viewing_activity = cleaned_datasets["viewing_activity"]
payments = cleaned_datasets["payments"]
support_tickets = cleaned_datasets["support_tickets"]
customer_feedback = cleaned_datasets["customer_feedback"]
churn_labels = cleaned_datasets["churn_labels"]

print("All datasets loaded successfully!")

for name, df in cleaned_datasets.items():
    print(name, df.shape)

All datasets loaded successfully!
customers (8000, 12)
subscription_plans (3, 6)
subscriptions (8875, 10)
content (500, 9)
viewing_activity (66422, 9)
payments (110735, 8)
support_tickets (6120, 11)
customer_feedback (5046, 6)
churn_labels (8000, 4)


In [107]:
# Step 47.7 – Filter Support Tickets as of May 31, 2026.
support_asof = support_tickets.copy()

support_asof["ticket_date"] = pd.to_datetime(
    support_asof["ticket_date"],
    errors="coerce"
)

support_asof = support_asof[
    support_asof["ticket_date"].notna()
    & (support_asof["ticket_date"] <= prediction_cutoff_date)
    & (support_asof["customer_id"].isin(eligible_cohort["customer_id"]))
].copy()

print("Support tickets:", len(support_asof))
print("Unique customers:", support_asof["customer_id"].nunique())
print("Earliest ticket date:", support_asof["ticket_date"].min())
print("Latest ticket date:", support_asof["ticket_date"].max())
print("Records after cutoff:", (support_asof["ticket_date"] > prediction_cutoff_date).sum())
print("Missing ticket dates:", support_asof["ticket_date"].isna().sum())

Support tickets: 4407
Unique customers: 3296
Earliest ticket date: 2019-03-03 00:00:00
Latest ticket date: 2026-05-31 00:00:00
Records after cutoff: 0
Missing ticket dates: 0


In [108]:
# Step 47.8 – Aggregate Support Ticket Features
customer_support_asof = (
    support_asof
    .groupby("customer_id")
    .agg(
        total_support_tickets=("ticket_id", "nunique"),
        unique_issue_categories=("issue_category", "nunique"),
        most_common_issue_category=(
            "issue_category",
            lambda x: x.mode().iloc[0]
        )
    )
    .reset_index()
)

print("Support features shape:", customer_support_asof.shape)
print("Unique customers:", customer_support_asof["customer_id"].nunique())
print("Total support tickets:", customer_support_asof["total_support_tickets"].sum())
print("Duplicate customer IDs:", customer_support_asof["customer_id"].duplicated().sum())

print("\nMissing values:")
print(customer_support_asof.isna().sum())

Support features shape: (3296, 4)
Unique customers: 3296
Total support tickets: 4407
Duplicate customer IDs: 0

Missing values:
customer_id                   0
total_support_tickets         0
unique_issue_categories       0
most_common_issue_category    0
dtype: int64


In [109]:
# Step 47.9 – Merge Support Features
churn_features = churn_features.merge(
    customer_support_asof,
    on="customer_id",
    how="left",
    validate="one_to_one"
)

churn_features["has_support_activity"] = (
    churn_features["total_support_tickets"].notna().astype(int)
)

support_count_columns = [
    "total_support_tickets",
    "unique_issue_categories"
]

churn_features[support_count_columns] = (
    churn_features[support_count_columns].fillna(0)
)

print("Churn features shape:", churn_features.shape)
print("Customers with support activity:", churn_features["has_support_activity"].sum())
print("Customers without support activity:", (churn_features["has_support_activity"] == 0).sum())
print("Duplicate customer IDs:", churn_features["customer_id"].duplicated().sum())

print("\nMissing values:")
print(churn_features.isna().sum())

Churn features shape: (6260, 25)
Customers with support activity: 3296
Customers without support activity: 2964
Duplicate customer IDs: 0

Missing values:
customer_id                            0
first_name                             0
last_name                              0
age                                    0
gender                                 0
country                                0
city                                   0
acquisition_channel                    0
customer_segment                       0
preferred_language                     0
tenure_days                            0
total_viewing_sessions                 0
total_watch_duration_minutes           0
average_watch_duration_minutes        42
average_completion_percentage         42
average_session_duration_minutes      42
unique_content_watched                 0
active_viewing_days                    0
last_viewing_date                     42
days_since_last_viewing               42
has_viewing_activity     

In [110]:
print("Current shape:", churn_features.shape)

print("\nCurrent columns:")
print(churn_features.columns.tolist())

print("\nPayment features available:",
      "customer_payment_asof" in globals())

if "customer_payment_asof" in globals():
    print("Payment features shape:", customer_payment_asof.shape)

Current shape: (6260, 25)

Current columns:
['customer_id', 'first_name', 'last_name', 'age', 'gender', 'country', 'city', 'acquisition_channel', 'customer_segment', 'preferred_language', 'tenure_days', 'total_viewing_sessions', 'total_watch_duration_minutes', 'average_watch_duration_minutes', 'average_completion_percentage', 'average_session_duration_minutes', 'unique_content_watched', 'active_viewing_days', 'last_viewing_date', 'days_since_last_viewing', 'has_viewing_activity', 'total_support_tickets', 'unique_issue_categories', 'most_common_issue_category', 'has_support_activity']

Payment features available: True
Payment features shape: (6260, 12)


In [111]:
churn_features = churn_features.merge(
    customer_payment_asof,
    on="customer_id",
    how="left",
    validate="one_to_one"
)

churn_features["has_successful_payment"] = (
    churn_features["successful_payment_count"] > 0
).astype(int)

print("Churn features shape:", churn_features.shape)
print("Customers with successful payments:",
      churn_features["has_successful_payment"].sum())
print("Customers without successful payments:",
      (churn_features["has_successful_payment"] == 0).sum())
print("Duplicate customer IDs:",
      churn_features["customer_id"].duplicated().sum())

Churn features shape: (6260, 37)
Customers with successful payments: 6253
Customers without successful payments: 7
Duplicate customer IDs: 0


In [112]:
# Step 47.10 – Filter customer feedback
feedback_asof = customer_feedback.copy()

feedback_asof["feedback_date"] = pd.to_datetime(
    feedback_asof["feedback_date"], errors="coerce"
)

feedback_asof = feedback_asof[
    feedback_asof["feedback_date"].notna()
    & (feedback_asof["feedback_date"] <= prediction_cutoff_date)
    & (feedback_asof["customer_id"].isin(eligible_cohort["customer_id"]))
].copy()

print("Feedback records:", len(feedback_asof))
print("Unique customers:", feedback_asof["customer_id"].nunique())
print("Date range:",
      feedback_asof["feedback_date"].min(),
      "to",
      feedback_asof["feedback_date"].max())

print("\nSentiment distribution:")
print(feedback_asof["sentiment"].value_counts())

print("\nMissing values:")
print(feedback_asof.isnull().sum())

Feedback records: 3718
Unique customers: 2669
Date range: 2019-03-06 00:00:00 to 2026-05-31 00:00:00

Sentiment distribution:


KeyError: 'sentiment'

In [113]:
print("Feedback columns:")
print(feedback_asof.columns.tolist())

print("\nDataset sample:")
display(feedback_asof.head())

Feedback columns:
['feedback_id', 'customer_id', 'feedback_date', 'rating', 'feedback_text', 'sentiment_label']

Dataset sample:


,feedback_id,customer_id,feedback_date,rating,feedback_text,sentiment_label
0,FB0000001,CUST000001,2025-12-25,1,"Not enough new content being added, feels like...",Negative
1,FB0000002,CUST000001,2022-08-12,5,"Great value for the price, my whole family use...",Positive
2,FB0000003,CUST000002,2026-04-11,1,Customer support took too long to respond to m...,Negative
4,FB0000005,CUST000008,2024-05-30,3,The service is decent but the content library ...,Neutral
5,FB0000006,CUST000009,2026-03-14,4,"Great value for the price, my whole family use...",Positive


In [114]:
# Step 47.10 – Correct the validation
print("Feedback records:", len(feedback_asof))
print("Unique customers:", feedback_asof["customer_id"].nunique())

print("\nSentiment distribution:")
print(feedback_asof["sentiment_label"].value_counts())

print("\nRating distribution:")
print(feedback_asof["rating"].value_counts().sort_index())

print("\nMissing values:")
print(feedback_asof.isnull().sum())

Feedback records: 3718
Unique customers: 2669

Sentiment distribution:
sentiment_label
Positive    1518
Neutral     1193
Negative    1007
Name: count, dtype: int64

Rating distribution:
rating
1     502
2     488
3    1185
4     786
5     757
Name: count, dtype: int64

Missing values:
feedback_id        0
customer_id        0
feedback_date      0
rating             0
feedback_text      0
sentiment_label    0
dtype: int64


In [115]:
# Step 47.11 – Create customer-level feedback features
feedback_asof["is_positive_feedback"] = (
    feedback_asof["sentiment_label"] == "Positive"
).astype(int)

feedback_asof["is_neutral_feedback"] = (
    feedback_asof["sentiment_label"] == "Neutral"
).astype(int)

feedback_asof["is_negative_feedback"] = (
    feedback_asof["sentiment_label"] == "Negative"
).astype(int)

customer_feedback_asof = (
    feedback_asof.groupby("customer_id")
    .agg(
        total_feedback_count=("feedback_id", "nunique"),
        average_feedback_rating=("rating", "mean"),
        positive_feedback_count=("is_positive_feedback", "sum"),
        neutral_feedback_count=("is_neutral_feedback", "sum"),
        negative_feedback_count=("is_negative_feedback", "sum"),
        negative_feedback_rate=("is_negative_feedback", "mean"),
        last_feedback_date=("feedback_date", "max")
    )
    .reset_index()
)

latest_feedback = (
    feedback_asof.sort_values(["feedback_date", "feedback_id"])
    .groupby("customer_id")
    .tail(1)[["customer_id", "rating", "sentiment_label"]]
    .rename(columns={
        "rating": "latest_feedback_rating",
        "sentiment_label": "latest_feedback_sentiment"
    })
)

customer_feedback_asof = customer_feedback_asof.merge(
    latest_feedback,
    on="customer_id",
    how="left",
    validate="one_to_one"
)

customer_feedback_asof["days_since_last_feedback"] = (
    prediction_cutoff_date - customer_feedback_asof["last_feedback_date"]
).dt.days

print("Feedback features shape:", customer_feedback_asof.shape)
print("Unique customers:", customer_feedback_asof["customer_id"].nunique())
print("Duplicate customer IDs:", customer_feedback_asof["customer_id"].duplicated().sum())

print("\nMissing values:")
print(customer_feedback_asof.isnull().sum())

display(customer_feedback_asof.head())

Feedback features shape: (2669, 11)
Unique customers: 2669
Duplicate customer IDs: 0

Missing values:
customer_id                  0
total_feedback_count         0
average_feedback_rating      0
positive_feedback_count      0
neutral_feedback_count       0
negative_feedback_count      0
negative_feedback_rate       0
last_feedback_date           0
latest_feedback_rating       0
latest_feedback_sentiment    0
days_since_last_feedback     0
dtype: int64


,customer_id,total_feedback_count,average_feedback_rating,positive_feedback_count,neutral_feedback_count,negative_feedback_count,negative_feedback_rate,last_feedback_date,latest_feedback_rating,latest_feedback_sentiment,days_since_last_feedback
0,CUST000001,2,3.0,1,0,1,0.5,2025-12-25,1,Negative,157
1,CUST000002,1,1.0,0,0,1,1.0,2026-04-11,1,Negative,50
2,CUST000008,1,3.0,0,1,0,0.0,2024-05-30,3,Neutral,731
3,CUST000009,1,4.0,1,0,0,0.0,2026-03-14,4,Positive,78
4,CUST000013,1,4.0,1,0,0,0.0,2024-07-25,4,Positive,675


In [116]:
# Step 47.12 – Merge customer feedback features
churn_features = churn_features.merge(
    customer_feedback_asof,
    on="customer_id",
    how="left",
    validate="one_to_one"
)

churn_features["has_feedback_activity"] = (
    churn_features["total_feedback_count"].notna().astype(int)
)

feedback_count_columns = [
    "total_feedback_count",
    "positive_feedback_count",
    "neutral_feedback_count",
    "negative_feedback_count",
    "negative_feedback_rate"
]

churn_features[feedback_count_columns] = (
    churn_features[feedback_count_columns].fillna(0)
)

print("Churn features shape:", churn_features.shape)
print("Customers with feedback:",
      churn_features["has_feedback_activity"].sum())
print("Customers without feedback:",
      (churn_features["has_feedback_activity"] == 0).sum())
print("Duplicate customer IDs:",
      churn_features["customer_id"].duplicated().sum())

print("\nMissing values:")
print(churn_features.isnull().sum().sort_values(ascending=False).head(10))

Churn features shape: (6260, 48)
Customers with feedback: 2669
Customers without feedback: 3591
Duplicate customer IDs: 0

Missing values:
latest_feedback_rating              3591
latest_feedback_sentiment           3591
days_since_last_feedback            3591
average_feedback_rating             3591
last_feedback_date                  3591
most_common_issue_category          2964
average_session_duration_minutes      42
average_watch_duration_minutes        42
days_since_last_viewing               42
last_viewing_date                     42
dtype: int64


In [117]:
# Step 48.1 – Inspect the dataset columns
print("Viewing activity columns:")
print(viewing_asof.columns.tolist())

print("\nContent dataset columns:")
print(content.columns.tolist())

print("\nContent dataset shape:", content.shape)

print("\nContent sample:")
display(content.head())

print("\nViewing activity sample:")
display(viewing_asof.head())

Viewing activity columns:
['viewing_id', 'customer_id', 'content_id', 'viewing_date', 'watch_duration_minutes', 'completion_percentage', 'device_type', 'login_location', 'session_duration_minutes']

Content dataset columns:
['content_id', 'title', 'content_type', 'genre', 'release_year', 'duration_minutes', 'content_language', 'production_country', 'maturity_rating']

Content dataset shape: (500, 9)

Content sample:


,content_id,title,content_type,genre,release_year,duration_minutes,content_language,production_country,maturity_rating
0,CNT00001,Golden Horizon,TV Show,Horror,2021,51,English,India,TV-14
1,CNT00002,Restless Kingdom,TV Show,Comedy,2004,39,English,United States,TV-MA
2,CNT00003,Silent Symphony,Movie,Family,2016,117,English,India,NR
3,CNT00004,Silent Legacy,TV Show,Stand-Up Comedy,2004,72,Indonesian,India,PG-13
4,CNT00005,Golden Kingdom,Movie,Reality,2017,100,English,Philippines,G



Viewing activity sample:


,viewing_id,customer_id,content_id,viewing_date,watch_duration_minutes,completion_percentage,device_type,login_location,session_duration_minutes
0,VIEW00000001,CUST000001,CNT00299,2025-08-14,14,100.0,Smart TV,United States,31
1,VIEW00000002,CUST000001,CNT00098,2026-03-07,42,49.0,Smart TV,United States,51
2,VIEW00000003,CUST000001,CNT00187,2026-02-21,50,100.0,Tablet,United States,57
3,VIEW00000004,CUST000001,CNT00126,2026-05-18,71,68.0,Smart TV,United States,83
4,VIEW00000005,CUST000001,CNT00179,2026-05-12,49,69.0,Mobile,United States,64


In [118]:
# Step 48.2 – Join viewing activity with content details
viewing_content_asof = viewing_asof.merge(
    content[
        [
            "content_id",
            "content_type",
            "genre",
            "content_language",
            "maturity_rating"
        ]
    ],
    on="content_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

print("Joined dataset shape:", viewing_content_asof.shape)

print("\nContent matching:")
print(viewing_content_asof["_merge"].value_counts())

print("\nMissing content attributes:")
print(
    viewing_content_asof[
        ["content_type", "genre", "content_language", "maturity_rating"]
    ].isnull().sum()
)

print("\nSample joined data:")
display(viewing_content_asof.head())

Joined dataset shape: (45107, 14)

Content matching:
_merge
both          45107
left_only         0
right_only        0
Name: count, dtype: int64

Missing content attributes:
content_type        0
genre               0
content_language    0
maturity_rating     0
dtype: int64

Sample joined data:


,viewing_id,customer_id,content_id,viewing_date,watch_duration_minutes,completion_percentage,device_type,login_location,session_duration_minutes,content_type,genre,content_language,maturity_rating,_merge
0,VIEW00000001,CUST000001,CNT00299,2025-08-14,14,100.0,Smart TV,United States,31,Movie,Comedy,English,NR,both
1,VIEW00000002,CUST000001,CNT00098,2026-03-07,42,49.0,Smart TV,United States,51,Movie,Thriller,Spanish,PG,both
2,VIEW00000003,CUST000001,CNT00187,2026-02-21,50,100.0,Tablet,United States,57,TV Show,Action,Hindi,TV-14,both
3,VIEW00000004,CUST000001,CNT00126,2026-05-18,71,68.0,Smart TV,United States,83,Stand-Up,Anime,Korean,NR,both
4,VIEW00000005,CUST000001,CNT00179,2026-05-12,49,69.0,Mobile,United States,64,Movie,Romance,Indonesian,PG,both


In [119]:
# Step 48.3 – Create customer-level content preference features
customer_content_preferences = (
    viewing_content_asof.groupby("customer_id")
    .agg(
        most_watched_genre=("genre", lambda x: x.mode().iloc[0]),
        unique_genres_watched=("genre", "nunique"),
        most_watched_content_type=("content_type", lambda x: x.mode().iloc[0]),
        unique_content_types_watched=("content_type", "nunique"),
        most_watched_language=("content_language", lambda x: x.mode().iloc[0]),
        unique_languages_watched=("content_language", "nunique"),
        most_watched_maturity_rating=("maturity_rating", lambda x: x.mode().iloc[0]),
        unique_maturity_ratings_watched=("maturity_rating", "nunique")
    )
    .reset_index()
)

print("Content preference features shape:",
      customer_content_preferences.shape)

print("Unique customers:",
      customer_content_preferences["customer_id"].nunique())

print("Duplicate customer IDs:",
      customer_content_preferences["customer_id"].duplicated().sum())

print("\nMissing values:")
print(customer_content_preferences.isnull().sum())

display(customer_content_preferences.head())

Content preference features shape: (6218, 9)
Unique customers: 6218
Duplicate customer IDs: 0

Missing values:
customer_id                        0
most_watched_genre                 0
unique_genres_watched              0
most_watched_content_type          0
unique_content_types_watched       0
most_watched_language              0
unique_languages_watched           0
most_watched_maturity_rating       0
unique_maturity_ratings_watched    0
dtype: int64


,customer_id,most_watched_genre,unique_genres_watched,most_watched_content_type,unique_content_types_watched,most_watched_language,unique_languages_watched,most_watched_maturity_rating,unique_maturity_ratings_watched
0,CUST000001,Action,5,Movie,3,English,5,NR,3
1,CUST000002,Anime,5,Movie,2,English,3,R,4
2,CUST000004,Fantasy,6,Movie,3,English,3,TV-MA,4
3,CUST000005,Anime,7,TV Show,4,Hindi,5,PG-13,5
4,CUST000006,Anime,6,Movie,4,Hindi,6,PG,4


In [120]:
# Step 48.4 – Merge content preference features
churn_features = churn_features.merge(
    customer_content_preferences,
    on="customer_id",
    how="left",
    validate="one_to_one"
)

churn_features["has_content_preference_data"] = (
    churn_features["most_watched_genre"].notna().astype(int)
)

content_count_columns = [
    "unique_genres_watched",
    "unique_content_types_watched",
    "unique_languages_watched",
    "unique_maturity_ratings_watched"
]

churn_features[content_count_columns] = (
    churn_features[content_count_columns].fillna(0)
)

print("Churn features shape:", churn_features.shape)

print("Customers with content preferences:",
      churn_features["has_content_preference_data"].sum())

print("Customers without content preferences:",
      (churn_features["has_content_preference_data"] == 0).sum())

print("Duplicate customer IDs:",
      churn_features["customer_id"].duplicated().sum())

print("\nMissing values:")
print(churn_features.isnull().sum().sort_values(ascending=False).head(12))

Churn features shape: (6260, 57)
Customers with content preferences: 6218
Customers without content preferences: 42
Duplicate customer IDs: 0

Missing values:
average_feedback_rating             3591
latest_feedback_rating              3591
last_feedback_date                  3591
days_since_last_feedback            3591
latest_feedback_sentiment           3591
most_common_issue_category          2964
average_session_duration_minutes      42
days_since_last_viewing               42
average_watch_duration_minutes        42
most_watched_genre                    42
most_watched_maturity_rating          42
most_watched_language                 42
dtype: int64


In [121]:
# Step 49.1 – Create the 60-day viewing window
recent_30_start = prediction_cutoff_date - pd.Timedelta(days=29)
previous_30_start = prediction_cutoff_date - pd.Timedelta(days=59)

viewing_recent_60d = viewing_asof[
    (viewing_asof["viewing_date"] >= previous_30_start)
    & (viewing_asof["viewing_date"] <= prediction_cutoff_date)
].copy()

viewing_recent_60d["viewing_period"] = "previous_30d"

viewing_recent_60d.loc[
    viewing_recent_60d["viewing_date"] >= recent_30_start,
    "viewing_period"
] = "recent_30d"

print("Recent 30-day start:", recent_30_start)
print("Previous 30-day start:", previous_30_start)

print("\nViewing records:", len(viewing_recent_60d))

print("\nViewing period distribution:")
print(viewing_recent_60d["viewing_period"].value_counts())

print("\nUnique customers:",
      viewing_recent_60d["customer_id"].nunique())

print("\nDate range:")
print(
    viewing_recent_60d["viewing_date"].min(),
    "to",
    viewing_recent_60d["viewing_date"].max()
)

Recent 30-day start: 2026-05-02 00:00:00
Previous 30-day start: 2026-04-02 00:00:00

Viewing records: 10920

Viewing period distribution:
viewing_period
recent_30d      7379
previous_30d    3541
Name: count, dtype: int64

Unique customers: 4611

Date range:
2026-04-02 00:00:00 to 2026-05-31 00:00:00


In [122]:
# Step 49.2 – Aggregate viewing activity by period
customer_viewing_trend_summary = (
    viewing_recent_60d.groupby(["customer_id", "viewing_period"])
    .agg(
        viewing_sessions=("viewing_id", "nunique"),
        watch_duration_minutes=("watch_duration_minutes", "sum"),
        average_completion_percentage=("completion_percentage", "mean")
    )
    .unstack("viewing_period")
)

customer_viewing_trend_summary.columns = [
    f"{feature}_{period}"
    for feature, period in customer_viewing_trend_summary.columns
]

customer_viewing_trend_summary = (
    customer_viewing_trend_summary.reset_index()
)

count_columns = [
    "viewing_sessions_previous_30d",
    "viewing_sessions_recent_30d",
    "watch_duration_minutes_previous_30d",
    "watch_duration_minutes_recent_30d"
]

customer_viewing_trend_summary[count_columns] = (
    customer_viewing_trend_summary[count_columns].fillna(0)
)

print("Viewing trend features shape:",
      customer_viewing_trend_summary.shape)

print("Unique customers:",
      customer_viewing_trend_summary["customer_id"].nunique())

print("Duplicate customer IDs:",
      customer_viewing_trend_summary["customer_id"].duplicated().sum())

print("\nMissing values:")
print(customer_viewing_trend_summary.isnull().sum())

display(customer_viewing_trend_summary.head())

Viewing trend features shape: (4611, 7)
Unique customers: 4611
Duplicate customer IDs: 0

Missing values:
customer_id                                      0
viewing_sessions_previous_30d                    0
viewing_sessions_recent_30d                      0
watch_duration_minutes_previous_30d              0
watch_duration_minutes_recent_30d                0
average_completion_percentage_previous_30d    2066
average_completion_percentage_recent_30d      1104
dtype: int64


,customer_id,viewing_sessions_previous_30d,viewing_sessions_recent_30d,watch_duration_minutes_previous_30d,watch_duration_minutes_recent_30d,average_completion_percentage_previous_30d,average_completion_percentage_recent_30d
0,CUST000001,0.0,2.0,0.0,120.0,NaN,68.5
1,CUST000002,3.0,0.0,116.0,0.0,67.333333,NaN
2,CUST000004,1.0,1.0,74.0,79.0,71.000000,71.0
3,CUST000005,1.0,1.0,36.0,61.0,77.000000,52.0
4,CUST000006,1.0,1.0,59.0,64.0,82.000000,73.0


In [123]:
# Step 49.3 – Calculate viewing behaviour changes
customer_viewing_trend_summary["viewing_sessions_change"] = (
    customer_viewing_trend_summary["viewing_sessions_recent_30d"]
    - customer_viewing_trend_summary["viewing_sessions_previous_30d"]
)

customer_viewing_trend_summary["watch_duration_change"] = (
    customer_viewing_trend_summary["watch_duration_minutes_recent_30d"]
    - customer_viewing_trend_summary["watch_duration_minutes_previous_30d"]
)

customer_viewing_trend_summary["average_completion_change"] = (
    customer_viewing_trend_summary["average_completion_percentage_recent_30d"]
    - customer_viewing_trend_summary["average_completion_percentage_previous_30d"]
)

customer_viewing_trend_summary["active_previous_30d"] = (
    customer_viewing_trend_summary["viewing_sessions_previous_30d"] > 0
).astype(int)

customer_viewing_trend_summary["active_recent_30d"] = (
    customer_viewing_trend_summary["viewing_sessions_recent_30d"] > 0
).astype(int)

print("Updated shape:", customer_viewing_trend_summary.shape)

print("\nViewing activity status:")
print(
    customer_viewing_trend_summary[
        ["active_previous_30d", "active_recent_30d"]
    ].value_counts()
)

print("\nTrend feature summary:")
display(customer_viewing_trend_summary.head(10))

Updated shape: (4611, 12)

Viewing activity status:
active_previous_30d  active_recent_30d
0                    1                    2066
1                    1                    1441
                     0                    1104
Name: count, dtype: int64

Trend feature summary:


,customer_id,viewing_sessions_previous_30d,viewing_sessions_recent_30d,watch_duration_minutes_previous_30d,watch_duration_minutes_recent_30d,average_completion_percentage_previous_30d,average_completion_percentage_recent_30d,viewing_sessions_change,watch_duration_change,average_completion_change,active_previous_30d,active_recent_30d
0,CUST000001,0.0,2.0,0.0,120.0,NaN,68.500000,2.0,120.0,NaN,0,1
1,CUST000002,3.0,0.0,116.0,0.0,67.333333,NaN,-3.0,-116.0,NaN,1,0
2,CUST000004,1.0,1.0,74.0,79.0,71.000000,71.000000,0.0,5.0,0.000000,1,1
3,CUST000005,1.0,1.0,36.0,61.0,77.000000,52.000000,0.0,25.0,-25.000000,1,1
4,CUST000006,1.0,1.0,59.0,64.0,82.000000,73.000000,0.0,5.0,-9.000000,1,1
5,CUST000008,0.0,5.0,0.0,237.0,NaN,56.400000,5.0,237.0,NaN,0,1
6,CUST000009,2.0,2.0,94.0,152.0,71.500000,80.000000,0.0,58.0,8.500000,1,1
7,CUST000010,2.0,2.0,85.0,105.0,69.500000,58.000000,0.0,20.0,-11.500000,1,1
8,CUST000011,1.0,0.0,53.0,0.0,82.000000,NaN,-1.0,-53.0,NaN,1,0
9,CUST000012,1.0,3.0,59.0,135.0,79.000000,65.666667,2.0,76.0,-13.333333,1,1


In [124]:
# Step 49.3 – Final validation
print("Missing values:")
print(customer_viewing_trend_summary.isnull().sum())

print("\nViewing session totals:")
print(
    customer_viewing_trend_summary[
        [
            "viewing_sessions_previous_30d",
            "viewing_sessions_recent_30d"
        ]
    ].describe()
)

print("\nCustomers with no activity in either period:",
      (
          (customer_viewing_trend_summary["active_previous_30d"] == 0)
          & (customer_viewing_trend_summary["active_recent_30d"] == 0)
      ).sum())

Missing values:
customer_id                                      0
viewing_sessions_previous_30d                    0
viewing_sessions_recent_30d                      0
watch_duration_minutes_previous_30d              0
watch_duration_minutes_recent_30d                0
average_completion_percentage_previous_30d    2066
average_completion_percentage_recent_30d      1104
viewing_sessions_change                          0
watch_duration_change                            0
average_completion_change                     3170
active_previous_30d                              0
active_recent_30d                                0
dtype: int64

Viewing session totals:
       viewing_sessions_previous_30d  viewing_sessions_recent_30d
count                    4611.000000                  4611.000000
mean                        0.767946                     1.600304
std                         0.869301                     1.539245
min                         0.000000                     0.000000
25% 

In [125]:
# Step 49.4 – Merge viewing trends
churn_features = churn_features.merge(
    customer_viewing_trend_summary,
    on="customer_id",
    how="left",
    validate="one_to_one"
)

churn_features["has_viewing_activity_last_60d"] = (
    churn_features[
        [
            "viewing_sessions_previous_30d",
            "viewing_sessions_recent_30d"
        ]
    ].notna().any(axis=1).astype(int)
)

trend_zero_columns = [
    "viewing_sessions_previous_30d",
    "viewing_sessions_recent_30d",
    "watch_duration_minutes_previous_30d",
    "watch_duration_minutes_recent_30d",
    "viewing_sessions_change",
    "watch_duration_change",
    "active_previous_30d",
    "active_recent_30d"
]

churn_features[trend_zero_columns] = (
    churn_features[trend_zero_columns].fillna(0)
)

print("Shape:", churn_features.shape)
print("\nViewing activity in the last 60 days:")
print(churn_features["has_viewing_activity_last_60d"].value_counts())

print("\nDuplicate customer IDs:", churn_features["customer_id"].duplicated().sum())

print("\nMissing values in count and activity columns:")
print(churn_features[trend_zero_columns].isnull().sum())

print("\nCompletion-related missing values:")
print(
    churn_features[
        [
            "average_completion_percentage_previous_30d",
            "average_completion_percentage_recent_30d",
            "average_completion_change"
        ]
    ].isnull().sum()
)

Shape: (6260, 69)

Viewing activity in the last 60 days:
has_viewing_activity_last_60d
1    4611
0    1649
Name: count, dtype: int64

Duplicate customer IDs: 0

Missing values in count and activity columns:
viewing_sessions_previous_30d          0
viewing_sessions_recent_30d            0
watch_duration_minutes_previous_30d    0
watch_duration_minutes_recent_30d      0
viewing_sessions_change                0
watch_duration_change                  0
active_previous_30d                    0
active_recent_30d                      0
dtype: int64

Completion-related missing values:
average_completion_percentage_previous_30d    3715
average_completion_percentage_recent_30d      2753
average_completion_change                     4819
dtype: int64


In [126]:
# Step 50 – Customer Engagement Score
engagement_components = pd.DataFrame(index=churn_features.index)

engagement_components["viewing_frequency_score"] = (
    churn_features["total_viewing_sessions"].rank(pct=True)
)

engagement_components["watch_time_score"] = (
    churn_features["total_watch_duration_minutes"].rank(pct=True)
)

engagement_components["active_days_score"] = (
    churn_features["active_viewing_days"].rank(pct=True)
)

engagement_components["viewing_recency_score"] = (
    churn_features["days_since_last_viewing"].rank(
        pct=True,
        ascending=False
    ).fillna(0)
)

churn_features["customer_engagement_score"] = (
    engagement_components.mean(axis=1) * 100
    * churn_features["has_viewing_activity"]
)

churn_features["customer_engagement_score"] = (
    churn_features["customer_engagement_score"].round(2)
)

print("Shape:", churn_features.shape)
print("\nEngagement score statistics:")
print(churn_features["customer_engagement_score"].describe())

print("\nCustomers with zero engagement score:",
      (churn_features["customer_engagement_score"] == 0).sum())

print("\nMissing engagement scores:",
      churn_features["customer_engagement_score"].isna().sum())

Shape: (6260, 70)

Engagement score statistics:
count    6260.000000
mean       49.922441
std        23.387032
min         0.000000
25%        32.810000
50%        50.710000
75%        68.702500
max        99.230000
Name: customer_engagement_score, dtype: float64

Customers with zero engagement score: 42

Missing engagement scores: 0


In [127]:
# Step 51 – Historical Subscription Features
print("Subscription columns:")
print(subscriptions.columns.tolist())

print("\nData types:")
print(subscriptions.dtypes)

print("\nFirst 10 records:")
display(subscriptions.head(10))

print("\nSubscription status distribution:")
print(subscriptions["subscription_status"].value_counts(dropna=False))

Subscription columns:
['subscription_id', 'customer_id', 'plan_id', 'subscription_start_date', 'subscription_end_date', 'subscription_status', 'auto_renew', 'cancellation_date', 'cancellation_reason', 'monthly_price']

Data types:
subscription_id                str
customer_id                    str
plan_id                        str
subscription_start_date        str
subscription_end_date          str
subscription_status            str
auto_renew                    bool
cancellation_date              str
cancellation_reason            str
monthly_price              float64
dtype: object

First 10 records:


,subscription_id,customer_id,plan_id,subscription_start_date,subscription_end_date,subscription_status,auto_renew,cancellation_date,cancellation_reason,monthly_price
0,SUB0000001,CUST000001,PLAN03,2022-04-30,NaN,Active,True,NaN,NaN,22.99
1,SUB0000002,CUST000002,PLAN02,2025-12-24,NaN,Active,True,NaN,NaN,15.49
2,SUB0000003,CUST000003,PLAN03,2019-12-15,2026-04-20,Cancelled,True,2026-04-20,Too expensive,22.99
3,SUB0000004,CUST000004,PLAN01,2019-01-28,NaN,Active,False,NaN,NaN,6.99
4,SUB0000005,CUST000005,PLAN01,2024-07-24,NaN,Active,True,NaN,NaN,6.99
5,SUB0000006,CUST000006,PLAN03,2025-07-19,NaN,Active,True,NaN,NaN,22.99
6,SUB0000007,CUST000007,PLAN02,2025-10-30,2026-05-05,Cancelled,True,2026-05-05,Password sharing crackdown,15.49
7,SUB0000008,CUST000008,PLAN02,2023-08-21,NaN,Active,True,NaN,NaN,15.49
8,SUB0000009,CUST000009,PLAN02,2019-11-18,NaN,Active,True,NaN,NaN,15.49
9,SUB0000010,CUST000010,PLAN01,2021-12-03,NaN,Active,True,NaN,NaN,6.99



Subscription status distribution:
subscription_status
Active          6009
Cancelled       1991
Plan Changed     875
Name: count, dtype: int64


In [128]:
# Step 51.1 – Inspect subscription history
subscription_check = subscriptions.copy()

date_columns = [
    "subscription_start_date",
    "subscription_end_date",
    "cancellation_date"
]

for col in date_columns:
    subscription_check[col] = pd.to_datetime(
        subscription_check[col], errors="coerce"
    )

print("Dataset shape:", subscription_check.shape)

print("\nUnique customers:",
      subscription_check["customer_id"].nunique())

print("\nDuplicate subscription IDs:",
      subscription_check["subscription_id"].duplicated().sum())

print("\nMissing values:")
print(subscription_check.isnull().sum())

print("\nDate ranges:")
for col in date_columns:
    print(
        col,
        subscription_check[col].min(),
        "to",
        subscription_check[col].max()
    )

print("\nSubscriptions started after cutoff:",
      (subscription_check["subscription_start_date"] > prediction_cutoff_date).sum())

print("\nSubscriptions ending after cutoff:",
      (subscription_check["subscription_end_date"] > prediction_cutoff_date).sum())

print("\nSubscriptions per customer:")
print(subscription_check.groupby("customer_id").size().value_counts().sort_index())

display(subscription_check.head(10))

Dataset shape: (8875, 10)

Unique customers: 8000

Duplicate subscription IDs: 0

Missing values:
subscription_id               0
customer_id                   0
plan_id                       0
subscription_start_date       0
subscription_end_date      6051
subscription_status           0
auto_renew                    0
cancellation_date          6884
cancellation_reason        6884
monthly_price                 0
dtype: int64

Date ranges:
subscription_start_date 2019-01-01 00:00:00 to 2026-06-29 00:00:00
subscription_end_date 2019-04-11 00:00:00 to 2026-06-30 00:00:00
cancellation_date 2025-10-03 00:00:00 to 2026-06-30 00:00:00

Subscriptions started after cutoff: 29

Subscriptions ending after cutoff: 278

Subscriptions per customer:
1    7125
2     875
Name: count, dtype: int64


,subscription_id,customer_id,plan_id,subscription_start_date,subscription_end_date,subscription_status,auto_renew,cancellation_date,cancellation_reason,monthly_price
0,SUB0000001,CUST000001,PLAN03,2022-04-30,NaT,Active,True,NaT,NaN,22.99
1,SUB0000002,CUST000002,PLAN02,2025-12-24,NaT,Active,True,NaT,NaN,15.49
2,SUB0000003,CUST000003,PLAN03,2019-12-15,2026-04-20,Cancelled,True,2026-04-20,Too expensive,22.99
3,SUB0000004,CUST000004,PLAN01,2019-01-28,NaT,Active,False,NaT,NaN,6.99
4,SUB0000005,CUST000005,PLAN01,2024-07-24,NaT,Active,True,NaT,NaN,6.99
5,SUB0000006,CUST000006,PLAN03,2025-07-19,NaT,Active,True,NaT,NaN,22.99
6,SUB0000007,CUST000007,PLAN02,2025-10-30,2026-05-05,Cancelled,True,2026-05-05,Password sharing crackdown,15.49
7,SUB0000008,CUST000008,PLAN02,2023-08-21,NaT,Active,True,NaT,NaN,15.49
8,SUB0000009,CUST000009,PLAN02,2019-11-18,NaT,Active,True,NaT,NaN,15.49
9,SUB0000010,CUST000010,PLAN01,2021-12-03,NaT,Active,True,NaT,NaN,6.99


In [129]:
# Step 51.2 – Check subscription date patterns
print("Subscription status:")
print(subscription_check["subscription_status"].value_counts(dropna=False))

print("\nEnd dates relative to cutoff:")
print(
    pd.Series({
        "Ended on or before cutoff": (
            subscription_check["subscription_end_date"] <= prediction_cutoff_date
        ).sum(),
        "Ended after cutoff": (
            subscription_check["subscription_end_date"] > prediction_cutoff_date
        ).sum(),
        "No end date": subscription_check["subscription_end_date"].isna().sum()
    })
)

print("\nCancellation dates relative to cutoff:")
print(
    pd.Series({
        "Cancelled on or before cutoff": (
            subscription_check["cancellation_date"] <= prediction_cutoff_date
        ).sum(),
        "Cancelled after cutoff": (
            subscription_check["cancellation_date"] > prediction_cutoff_date
        ).sum(),
        "No cancellation date": subscription_check["cancellation_date"].isna().sum()
    })
)

print("\nEnd date and cancellation date comparison:")
print(
    (subscription_check["subscription_end_date"] ==
     subscription_check["cancellation_date"]).value_counts(dropna=False)
)

print("\nSubscriptions with cancellation dates after cutoff:")
display(
    subscription_check.loc[
        subscription_check["cancellation_date"] > prediction_cutoff_date,
        [
            "customer_id",
            "subscription_start_date",
            "subscription_end_date",
            "subscription_status",
            "cancellation_date",
            "monthly_price"
        ]
    ].head(10)
)

Subscription status:
subscription_status
Active          6009
Cancelled       1991
Plan Changed     875
Name: count, dtype: int64

End dates relative to cutoff:
Ended on or before cutoff    2546
Ended after cutoff            278
No end date                  6051
dtype: int64

Cancellation dates relative to cutoff:
Cancelled on or before cutoff    1740
Cancelled after cutoff            251
No cancellation date             6884
dtype: int64

End date and cancellation date comparison:
False    6926
True     1949
Name: count, dtype: int64

Subscriptions with cancellation dates after cutoff:


,customer_id,subscription_start_date,subscription_end_date,subscription_status,cancellation_date,monthly_price
22,CUST000023,2026-04-13,2026-06-05,Cancelled,2026-06-05,15.49
72,CUST000066,2025-08-27,2026-06-09,Cancelled,2026-06-09,22.99
122,CUST000111,2025-09-14,2026-06-13,Cancelled,2026-06-13,15.49
146,CUST000133,2026-05-10,2026-06-25,Cancelled,2026-06-25,6.99
154,CUST000141,2025-07-23,2026-06-01,Cancelled,2026-06-01,6.99
199,CUST000180,2020-05-07,2026-06-04,Cancelled,2026-06-04,15.49
202,CUST000183,2020-04-12,2026-06-14,Cancelled,2026-06-14,6.99
207,CUST000187,2026-01-17,2026-06-19,Cancelled,2026-06-19,15.49
212,CUST000191,2023-11-11,2026-06-28,Cancelled,2026-06-28,22.99
350,CUST000312,2026-01-11,2026-06-08,Cancelled,2026-06-08,15.49


In [130]:
# Step 51.3 – Verify subscription history
subscription_check["end_date_period"] = np.select(
    [
        subscription_check["subscription_end_date"].isna(),
        subscription_check["subscription_end_date"] <= prediction_cutoff_date,
        subscription_check["subscription_end_date"] > prediction_cutoff_date
    ],
    ["No end date", "Ended by cutoff", "Ends after cutoff"],
    default="Unknown"
)

subscription_check["cancellation_period"] = np.select(
    [
        subscription_check["cancellation_date"].isna(),
        subscription_check["cancellation_date"] <= prediction_cutoff_date,
        subscription_check["cancellation_date"] > prediction_cutoff_date
    ],
    ["No cancellation", "Cancelled by cutoff", "Cancelled after cutoff"],
    default="Unknown"
)

print("Status vs. end-date period:")
display(pd.crosstab(
    subscription_check["subscription_status"],
    subscription_check["end_date_period"],
    margins=True
))

print("\nStatus vs. cancellation period:")
display(pd.crosstab(
    subscription_check["subscription_status"],
    subscription_check["cancellation_period"],
    margins=True
))

eligible_subscriptions = subscription_check[
    subscription_check["customer_id"].isin(eligible_cohort["customer_id"])
].copy()

print("\nEligible customers' subscription records:")
print(
    eligible_subscriptions.groupby("customer_id")
    .size()
    .value_counts()
    .sort_index()
)

active_asof = eligible_subscriptions[
    (eligible_subscriptions["subscription_start_date"] <= prediction_cutoff_date)
    & (
        eligible_subscriptions["subscription_end_date"].isna()
        | (eligible_subscriptions["subscription_end_date"] > prediction_cutoff_date)
    )
    & (
        eligible_subscriptions["cancellation_date"].isna()
        | (eligible_subscriptions["cancellation_date"] > prediction_cutoff_date)
    )
].copy()

print("\nSubscriptions active as of cutoff:")
print("Records:", len(active_asof))
print("Unique customers:", active_asof["customer_id"].nunique())
print("Duplicate customer records:", active_asof["customer_id"].duplicated().sum())

print("\nActive-as-of subscription status:")
print(active_asof["subscription_status"].value_counts(dropna=False))

Status vs. end-date period:


end_date_period,Ended by cutoff,Ends after cutoff,No end date,All
subscription_status,,,,
Active,0,0,6009,6009
Cancelled,1700,249,42,1991
Plan Changed,846,29,0,875
All,2546,278,6051,8875



Status vs. cancellation period:


cancellation_period,Cancelled after cutoff,Cancelled by cutoff,No cancellation,All
subscription_status,,,,
Active,0,0,6009,6009
Cancelled,251,1740,0,1991
Plan Changed,0,0,875,875
All,251,1740,6884,8875



Eligible customers' subscription records:
1    5585
2     675
Name: count, dtype: int64

Subscriptions active as of cutoff:
Records: 6260
Unique customers: 6260
Duplicate customer records: 0

Active-as-of subscription status:
subscription_status
Active          5989
Cancelled        249
Plan Changed      22
Name: count, dtype: int64


In [131]:
print("Eligible subscription records:", len(eligible_subscriptions))

print("\nRecords per eligible customer:")
print(
    eligible_subscriptions.groupby("customer_id")
    .size().value_counts().sort_index()
)

print("\nActive-as-of-cutoff records:", len(active_asof))
print("Unique active customers:", active_asof["customer_id"].nunique())
print("Duplicate customer records:", active_asof["customer_id"].duplicated().sum())

print("\nActive-as-of records by current status:")
print(active_asof["subscription_status"].value_counts())

print("\nCustomers in eligible cohort without an active subscription record:",
      len(set(eligible_cohort["customer_id"]) - set(active_asof["customer_id"])))

Eligible subscription records: 6935

Records per eligible customer:
1    5585
2     675
Name: count, dtype: int64

Active-as-of-cutoff records: 6260
Unique active customers: 6260
Duplicate customer records: 0

Active-as-of records by current status:
subscription_status
Active          5989
Cancelled        249
Plan Changed      22
Name: count, dtype: int64

Customers in eligible cohort without an active subscription record: 0


In [132]:
# Step 51.4 – Create historical subscription features
subscription_history_asof = eligible_subscriptions[
    eligible_subscriptions["subscription_start_date"] <= prediction_cutoff_date
].copy()

subscription_history_asof["ended_by_cutoff"] = (
    subscription_history_asof["subscription_end_date"].notna()
    & (subscription_history_asof["subscription_end_date"] <= prediction_cutoff_date)
).astype(int)

subscription_history_asof["cancelled_by_cutoff"] = (
    subscription_history_asof["cancellation_date"].notna()
    & (subscription_history_asof["cancellation_date"] <= prediction_cutoff_date)
).astype(int)

customer_subscription_history = (
    subscription_history_asof.groupby("customer_id")
    .agg(
        total_subscriptions_started=("subscription_id", "nunique"),
        previously_ended_subscription_count=("ended_by_cutoff", "sum"),
        previously_cancelled_subscription_count=("cancelled_by_cutoff", "sum"),
        unique_plans_used_asof=("plan_id", "nunique")
    )
    .reset_index()
)

customer_subscription_history["has_used_multiple_plans_asof"] = (
    customer_subscription_history["unique_plans_used_asof"] > 1
).astype(int)

current_subscription_features = active_asof[
    [
        "customer_id",
        "plan_id",
        "monthly_price",
        "subscription_start_date"
    ]
].copy()

current_subscription_features = current_subscription_features.rename(
    columns={
        "plan_id": "current_plan_id_asof",
        "monthly_price": "monthly_price_asof"
    }
)

current_subscription_features["subscription_tenure_days_asof"] = (
    prediction_cutoff_date
    - current_subscription_features["subscription_start_date"]
).dt.days

current_subscription_features = current_subscription_features.drop(
    columns=["subscription_start_date"]
)

churn_features = churn_features.merge(
    customer_subscription_history,
    on="customer_id",
    how="left",
    validate="one_to_one"
)

churn_features = churn_features.merge(
    current_subscription_features,
    on="customer_id",
    how="left",
    validate="one_to_one"
)

print("Final shape:", churn_features.shape)

print("\nNew subscription features:")
print(customer_subscription_history.columns.tolist())
print(current_subscription_features.columns.tolist())

print("\nSubscription feature summary:")
display(churn_features[
    [
        "total_subscriptions_started",
        "previously_ended_subscription_count",
        "previously_cancelled_subscription_count",
        "unique_plans_used_asof",
        "monthly_price_asof",
        "subscription_tenure_days_asof"
    ]
].describe())

print("\nMissing values in new features:")
print(churn_features[
    [
        "total_subscriptions_started",
        "previously_ended_subscription_count",
        "previously_cancelled_subscription_count",
        "unique_plans_used_asof",
        "has_used_multiple_plans_asof",
        "current_plan_id_asof",
        "monthly_price_asof",
        "subscription_tenure_days_asof"
    ]
].isnull().sum())

print("\nDuplicate customer IDs:", churn_features["customer_id"].duplicated().sum())

Final shape: (6260, 78)

New subscription features:
['customer_id', 'total_subscriptions_started', 'previously_ended_subscription_count', 'previously_cancelled_subscription_count', 'unique_plans_used_asof', 'has_used_multiple_plans_asof']
['customer_id', 'current_plan_id_asof', 'monthly_price_asof', 'subscription_tenure_days_asof']

Subscription feature summary:


,total_subscriptions_started,previously_ended_subscription_count,previously_cancelled_subscription_count,unique_plans_used_asof,monthly_price_asof,subscription_tenure_days_asof
count,6260.000000,6260.000000,6260.0,6260.000000,6260.000000,6260.000000
mean,1.104313,0.104313,0.0,1.068690,13.509249,1291.052236
std,0.305691,0.305691,0.0,0.252947,6.135752,788.430646
min,1.000000,0.000000,0.0,1.000000,6.990000,0.000000
25%,1.000000,0.000000,0.0,1.000000,6.990000,600.000000
50%,1.000000,0.000000,0.0,1.000000,15.490000,1287.500000
75%,1.000000,0.000000,0.0,1.000000,15.490000,1954.000000
max,2.000000,1.000000,0.0,2.000000,22.990000,2707.000000



Missing values in new features:
total_subscriptions_started                0
previously_ended_subscription_count        0
previously_cancelled_subscription_count    0
unique_plans_used_asof                     0
has_used_multiple_plans_asof               0
current_plan_id_asof                       0
monthly_price_asof                         0
subscription_tenure_days_asof              0
dtype: int64

Duplicate customer IDs: 0


In [133]:
# Step 52 – Integrate the churn target
churn_model_data = churn_features.merge(
    churn_target,
    on="customer_id",
    how="inner",
    validate="one_to_one"
)

print("Final dataset shape:", churn_model_data.shape)

print("\nTarget distribution:")
print(churn_model_data["churned"].value_counts())

print("\nTarget percentages:")
print(
    churn_model_data["churned"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nMissing target values:",
      churn_model_data["churned"].isna().sum())

print("\nDuplicate customer IDs:",
      churn_model_data["customer_id"].duplicated().sum())

print("\nFeature columns:", churn_model_data.shape[1] - 2)

Final dataset shape: (6260, 79)

Target distribution:
churned
False    6009
True      251
Name: count, dtype: int64

Target percentages:
churned
False    95.99
True      4.01
Name: proportion, dtype: float64

Missing target values: 0

Duplicate customer IDs: 0

Feature columns: 77


In [134]:
# Step 53 – Final data-quality and leakage audit
feature_columns = [
    col for col in churn_model_data.columns
    if col not in ["customer_id", "churned"]
]

missing_summary = (
    churn_model_data[feature_columns]
    .isnull()
    .sum()
    .sort_values(ascending=False)
)

numeric_columns = churn_model_data[feature_columns].select_dtypes(
    include="number"
).columns

infinite_counts = np.isinf(
    churn_model_data[numeric_columns]
).sum()

constant_columns = [
    col for col in feature_columns
    if churn_model_data[col].nunique(dropna=False) <= 1
]

date_columns = [
    col for col in feature_columns
    if "date" in col.lower()
]

leakage_candidates = [
    col for col in feature_columns
    if any(term in col.lower() for term in [
        "churn",
        "cancellation",
        "reason",
        "subscription_status",
        "auto_renew"
    ])
]

print("Dataset shape:", churn_model_data.shape)
print("Feature count:", len(feature_columns))
print("Duplicate customer IDs:",
      churn_model_data["customer_id"].duplicated().sum())

print("\nMissing values in features:")
print(missing_summary[missing_summary > 0])

print("\nInfinite values:")
print(infinite_counts[infinite_counts > 0])

print("\nConstant features:")
print(constant_columns)

print("\nDate-related feature columns:")
print(date_columns)

print("\nPotential leakage columns to review:")
print(leakage_candidates)

print("\nFeature data types:")
print(churn_model_data[feature_columns].dtypes.value_counts())

Dataset shape: (6260, 79)
Feature count: 77
Duplicate customer IDs: 0

Missing values in features:
average_completion_change                     4819
average_completion_percentage_previous_30d    3715
average_feedback_rating                       3591
latest_feedback_sentiment                     3591
latest_feedback_rating                        3591
days_since_last_feedback                      3591
last_feedback_date                            3591
most_common_issue_category                    2964
average_completion_percentage_recent_30d      2753
most_watched_maturity_rating                    42
most_watched_content_type                       42
most_watched_language                           42
average_watch_duration_minutes                  42
most_watched_genre                              42
average_completion_percentage                   42
days_since_last_viewing                         42
average_session_duration_minutes                42
last_viewing_date                 

In [135]:
# Step 53.1 – Complete the audit
print("Infinite values:")
print(infinite_counts[infinite_counts > 0])

print("\nConstant features:")
print(constant_columns)

print("\nDate-related columns:")
print(date_columns)

print("\nPotential leakage columns:")
print(leakage_candidates)

print("\nAll feature columns:")
print(feature_columns)

Infinite values:
Series([], dtype: int64)

Constant features:
['previously_cancelled_subscription_count']

Date-related columns:
['last_viewing_date', 'last_payment_date', 'last_successful_payment_date', 'last_feedback_date']

Potential leakage columns:
[]

All feature columns:
['first_name', 'last_name', 'age', 'gender', 'country', 'city', 'acquisition_channel', 'customer_segment', 'preferred_language', 'tenure_days', 'total_viewing_sessions', 'total_watch_duration_minutes', 'average_watch_duration_minutes', 'average_completion_percentage', 'average_session_duration_minutes', 'unique_content_watched', 'active_viewing_days', 'last_viewing_date', 'days_since_last_viewing', 'has_viewing_activity', 'total_support_tickets', 'unique_issue_categories', 'most_common_issue_category', 'has_support_activity', 'total_payment_transactions', 'successful_payment_count', 'failed_payment_count', 'total_amount_paid', 'average_successful_payment_amount', 'payment_success_rate', 'unique_payment_methods',

In [136]:
# Step 53.2 – Clean model features
churn_model_data["days_since_last_payment"] = (
    prediction_cutoff_date
    - pd.to_datetime(churn_model_data["last_payment_date"])
).dt.days

columns_to_drop = [
    "first_name",
    "last_name",
    "previously_cancelled_subscription_count",
    "last_viewing_date",
    "last_payment_date",
    "last_successful_payment_date",
    "last_feedback_date"
]

churn_model_data = churn_model_data.drop(columns=columns_to_drop)

print("Updated dataset shape:", churn_model_data.shape)

print("\nRemaining date columns:")
print(churn_model_data.select_dtypes(include=["datetime"]).columns.tolist())

print("\nMissing values in days_since_last_payment:",
      churn_model_data["days_since_last_payment"].isna().sum())

print("\nCategorical feature cardinality:")
categorical_columns = churn_model_data.select_dtypes(
    include=["object", "str"]
).columns

print(
    churn_model_data[categorical_columns]
    .nunique()
    .sort_values(ascending=False)
)

Updated dataset shape: (6260, 73)

Remaining date columns:
[]

Missing values in days_since_last_payment: 0

Categorical feature cardinality:
customer_id                     6260
city                              89
country                           21
most_watched_genre                16
preferred_language                14
most_watched_language             13
acquisition_channel                8
most_watched_maturity_rating       7
most_used_payment_method           7
most_common_issue_category         6
most_watched_content_type          5
customer_segment                   4
gender                             4
latest_feedback_sentiment          3
current_plan_id_asof               3
dtype: int64


In [137]:
# Step 53.3 – Inspect remaining missing values
model_features = [
    col for col in churn_model_data.columns
    if col not in ["customer_id", "churned"]
]

numeric_features = churn_model_data[model_features].select_dtypes(
    include="number"
).columns.tolist()

categorical_features = churn_model_data[model_features].select_dtypes(
    include=["object", "str", "category"]
).columns.tolist()

print("Numerical features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

print("\nMissing numerical values:")
print(
    churn_model_data[numeric_features]
    .isnull()
    .sum()
    .loc[lambda x: x > 0]
    .sort_values(ascending=False)
)

print("\nMissing categorical values:")
print(
    churn_model_data[categorical_features]
    .isnull()
    .sum()
    .loc[lambda x: x > 0]
    .sort_values(ascending=False)
)

Numerical features: 57
Categorical features: 14

Missing numerical values:
average_completion_change                     4819
average_completion_percentage_previous_30d    3715
average_feedback_rating                       3591
latest_feedback_rating                        3591
days_since_last_feedback                      3591
average_completion_percentage_recent_30d      2753
average_completion_percentage                   42
average_watch_duration_minutes                  42
days_since_last_viewing                         42
average_session_duration_minutes                42
average_successful_payment_amount                7
days_since_last_successful_payment               7
dtype: int64

Missing categorical values:
latest_feedback_sentiment       3591
most_common_issue_category      2964
most_watched_genre                42
most_watched_content_type         42
most_watched_language             42
most_watched_maturity_rating      42
dtype: int64


In [138]:
# Step 54 – Save the feature-engineered dataset
from pathlib import Path

project_root = Path.cwd().parent
final_data_dir = project_root / "data" / "processed" / "final"
final_data_dir.mkdir(parents=True, exist_ok=True)

output_path = final_data_dir / "churn_model_data.csv"

churn_model_data.to_csv(output_path, index=False)

saved_data = pd.read_csv(output_path)

print("Dataset saved successfully!")
print("File:", output_path)
print("Saved shape:", saved_data.shape)
print("Original shape:", churn_model_data.shape)
print("Missing values preserved:", saved_data.isnull().sum().sum())

Dataset saved successfully!
File: c:\Users\USER\Documents\Internmo Project1\fae5dd037f3e68b4f411b2243b81c23d\data\processed\final\churn_model_data.csv
Saved shape: (6260, 73)
Original shape: (6260, 73)
Missing values preserved: 28965
